# Data, Labels, CV & Sanity Guards

In [1]:
# ============================================================
# STAGE 1 — Data, Labels, CV & Sanity Guards (ONE CELL, FINAL+)
# - Lebih robust: auto-detect DATA_ROOT, validasi struktur, guard missing mask,
#   alignment sample_submission lebih ketat, sanity lebih informatif.
# - Tetap 1 jalur dan tidak mengubah output object utama yang dipakai stage berikutnya.
# ============================================================

import os, re, random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedKFold

try:
    import scipy.ndimage as ndi
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ----------------------------
# Helpers: safe prints
# ----------------------------
def _p(s): 
    print(s, flush=True)

# ----------------------------
# DATA_ROOT auto-detect (anti salah jalur)
# - Tetap pakai default kamu dulu.
# - Kalau tidak ada, cari folder di /kaggle/input yang punya sample_submission + train_images + test_images.
# ----------------------------
DEFAULT_ROOT = Path("/kaggle/input/recodai-luc-scientific-image-forgery-detection")

def auto_find_data_root(default_root: Path) -> Path:
    if default_root.exists():
        return default_root

    base = Path("/kaggle/input")
    if not base.exists():
        raise FileNotFoundError("Tidak menemukan /kaggle/input. Pastikan kamu run di Kaggle.")

    candidates = []
    for d in base.iterdir():
        if not d.is_dir():
            continue
        if (d / "sample_submission.csv").exists() and (d / "train_images").exists() and (d / "test_images").exists():
            candidates.append(d)

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"DATA_ROOT default tidak ada: {default_root}\n"
            "Dan tidak ada kandidat root yang punya (sample_submission.csv, train_images, test_images) di /kaggle/input.\n"
            "Cek: apakah kompetisi sudah di-Add Input?"
        )

    # pilih kandidat paling “lengkap”
    def score(d: Path) -> int:
        s = 0
        for need in ["train_masks", "supplemental_images", "supplemental_masks"]:
            if (d / need).exists():
                s += 1
        return s

    candidates.sort(key=score, reverse=True)
    if len(candidates) > 1:
        _p("[WARN] Ditemukan beberapa kandidat DATA_ROOT. Dipilih yang paling lengkap:")
        for c in candidates[:6]:
            _p(f"  - {c} (score={score(c)})")
    return candidates[0]

DATA_ROOT = auto_find_data_root(DEFAULT_ROOT)

# ----------------------------
# PATHS (sesuai struktur kompetisi)
# ----------------------------
PATHS = {
    "supp_images":  DATA_ROOT / "supplemental_images",
    "supp_masks":   DATA_ROOT / "supplemental_masks",
    "test_images":  DATA_ROOT / "test_images",
    "train_images": DATA_ROOT / "train_images",
    "train_masks":  DATA_ROOT / "train_masks",
    "sample_sub":   DATA_ROOT / "sample_submission.csv",
}

# DINO dir tetap sesuai konteks kamu
DINO_BASE_DIR = Path("/kaggle/input/dinov2/pytorch/base/1")

# Validasi minimal existence
for k, v in PATHS.items():
    if not v.exists():
        raise FileNotFoundError(f"PATH tidak ditemukan: {k} -> {v}")
if not DINO_BASE_DIR.exists():
    raise FileNotFoundError(f"DINOv2 base path tidak ditemukan: {DINO_BASE_DIR}")

_p("OK PATHS:")
_p(f"  {'data_root':>12}: {DATA_ROOT}")
for k, v in PATHS.items():
    _p(f"  {k:>12}: {v}")
_p(f"  {'dino_base':>12}: {DINO_BASE_DIR}")
_p(f"  scipy: {_HAS_SCIPY}")

# ----------------------------
# File indexing helpers
# ----------------------------
IMG_EXTS  = {".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp"}
MASK_EXTS = {".npy",".npz"} | IMG_EXTS  # tetap support kalau mask ternyata berupa image

def folder_stats(folder: Path, allow_ext=None, show=6):
    files = [p for p in folder.rglob("*") if p.is_file()]
    if allow_ext is not None:
        files = [p for p in files if p.suffix.lower() in allow_ext]
    exts = Counter([p.suffix.lower() for p in files])
    _p(f"\n[{folder}] files={len(files):,} ext_counts={dict(exts.most_common(8))}")
    for p in files[:show]:
        _p("  - " + str(p.relative_to(folder)))
    return files

def list_images(folder: Path):
    files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    files.sort()
    return files

def list_masks(folder: Path):
    files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in MASK_EXTS]
    files.sort()
    return files

def build_map_by_stem(files):
    mp = {}
    for p in files:
        if p.stem not in mp:
            mp[p.stem] = p
    return mp

# folder stats (biar gak salah jalur)
_ = folder_stats(PATHS["train_images"], allow_ext=IMG_EXTS)
_ = folder_stats(PATHS["train_masks"],  allow_ext=MASK_EXTS)
_ = folder_stats(PATHS["supp_images"],  allow_ext=IMG_EXTS)
_ = folder_stats(PATHS["supp_masks"],   allow_ext=MASK_EXTS)
_ = folder_stats(PATHS["test_images"],  allow_ext=IMG_EXTS)

# ----------------------------
# Index train images by variant (auth/forg)
# - Validasi subfolder authentic/forged kalau ada
# ----------------------------
train_files = list_images(PATHS["train_images"])

train_auth = {}
train_forg = {}

for p in train_files:
    cid = p.stem
    parts = [x.lower() for x in p.parts]
    if "authentic" in parts:
        train_auth.setdefault(cid, p)
    elif "forged" in parts:
        train_forg.setdefault(cid, p)
    else:
        # fallback: kalau struktur folder tidak eksplisit, abaikan dulu
        pass

if len(train_auth) == 0 and len(train_forg) == 0:
    # fallback last resort: coba deteksi dari path string
    _p("[WARN] Tidak menemukan folder 'authentic'/'forged' pada path parts. Coba fallback deteksi via string.")
    for p in train_files:
        s = str(p).lower()
        cid = p.stem
        if "authentic" in s:
            train_auth.setdefault(cid, p)
        elif "forged" in s:
            train_forg.setdefault(cid, p)

supp_files = list_images(PATHS["supp_images"])
test_files = list_images(PATHS["test_images"])
supp_map = build_map_by_stem(supp_files)
test_map = build_map_by_stem(test_files)

overlap = set(train_auth) & set(train_forg)
_p("\nIndexed images:")
_p(f"  train/authentic: {len(train_auth):,}")
_p(f"  train/forged   : {len(train_forg):,}")
_p(f"  overlap ids    : {len(overlap):,} (normal: pasangan auth+forg)")
_p(f"  supplemental   : {len(supp_map):,}")
_p(f"  test           : {len(test_map):,}")

# ----------------------------
# Index masks: group per case_id
# - Lebih aman untuk stem pattern: ambil token pertama sebelum '_' jika numeric
#   kalau tidak, fallback ke digit terpanjang
# ----------------------------
def mask_case_id_from_stem(stem: str) -> str:
    s = str(stem)
    tok0 = s.split("_")[0]
    if tok0.isdigit():
        return tok0
    nums = re.findall(r"\d+", s)
    if nums:
        nums = sorted(nums, key=len, reverse=True)
        return nums[0]
    return tok0

def group_masks(files):
    mp = defaultdict(list)
    for p in files:
        cid = str(mask_case_id_from_stem(p.stem))
        mp[cid].append(p)
    # sort deterministik per case_id
    out = {}
    for cid, lst in mp.items():
        lst2 = sorted(lst, key=lambda x: (x.suffix.lower(), x.name))
        out[cid] = lst2
    return out

train_mask_files = list_masks(PATHS["train_masks"])
supp_mask_files  = list_masks(PATHS["supp_masks"])
train_mask_map = group_masks(train_mask_files)
supp_mask_map  = group_masks(supp_mask_files)

_p("\nGrouped masks:")
_p(f"  train_masks groups: {len(train_mask_map):,} | files={len(train_mask_files):,}")
_p(f"  supp_masks  groups: {len(supp_mask_map):,} | files={len(supp_mask_files):,}")

def get_mask_list(cid: str, prefer_train=True):
    cid = str(cid)
    lst = []
    if prefer_train and cid in train_mask_map:
        lst += train_mask_map[cid]
    if (not prefer_train) and cid in supp_mask_map:
        lst += supp_mask_map[cid]
    if (not prefer_train) and cid in train_mask_map:
        lst += train_mask_map[cid]
    if prefer_train and cid in supp_mask_map:
        lst += supp_mask_map[cid]
    # unique by string path
    seen = set()
    out = []
    for p in lst:
        sp = str(p)
        if sp not in seen:
            out.append(p)
            seen.add(sp)
    return out

# ----------------------------
# Mask loader + ALIGN to image shape (robust)
# ----------------------------
def load_mask_any(path: str) -> np.ndarray:
    p = Path(path)
    ext = p.suffix.lower()
    if ext in IMG_EXTS:
        with Image.open(p) as im:
            return np.array(im.convert("L"))
    if ext == ".npy":
        return np.array(np.load(p, allow_pickle=False))
    if ext == ".npz":
        z = np.load(p)
        for k in z.files:
            a = z[k]
            if hasattr(a, "ndim") and a.ndim in (2, 3):
                return np.array(a)
        raise ValueError(f"NPZ tanpa array 2D/3D: {p}")
    raise ValueError(f"Mask ext tidak didukung: {ext} ({p})")

def to_binary_mask(arr: np.ndarray) -> np.ndarray:
    a = np.array(arr)
    # handle 3D: bisa (K,H,W) atau (H,W,K)
    if a.ndim == 3:
        # heuristik: jika channel kecil di akhir -> (H,W,K)
        if a.shape[-1] <= 8 and a.shape[0] > 8 and a.shape[1] > 8:
            a = np.max(a, axis=-1)
        else:
            a = np.max(a, axis=0)
    if np.issubdtype(a.dtype, np.floating):
        return (a > 0.5).astype(np.uint8)
    return (a > 0).astype(np.uint8)

def align_mask_to_image(mask2d: np.ndarray, H: int, W: int) -> np.ndarray:
    m = np.array(mask2d)
    if m.shape == (H, W):
        return m.astype(np.uint8)
    if m.shape == (W, H):
        return m.T.astype(np.uint8)
    if m.size == H * W:
        try:
            return m.reshape(H, W).astype(np.uint8)
        except Exception:
            pass
    # resize nearest (paling aman)
    im = Image.fromarray((m > 0).astype(np.uint8) * 255)
    im = im.resize((W, H), resample=Image.NEAREST)
    return (np.array(im) > 0).astype(np.uint8)

def union_masks_aligned(mask_paths: list, H: int, W: int) -> np.ndarray:
    if not mask_paths:
        return None
    out = np.zeros((H, W), dtype=np.uint8)
    for p in mask_paths:
        raw = load_mask_any(p)
        bm  = to_binary_mask(raw)
        am  = align_mask_to_image(bm, H, W)
        out = np.maximum(out, am)
    return out

# ----------------------------
# Build df_train_all (SAMPLE-LEVEL, UNIQUE per sample_id)
# ----------------------------
rows = []

# train/auth
for cid, p in train_auth.items():
    rows.append({
        "sample_id": f"{cid}__auth",
        "case_id": str(cid),
        "variant": "auth",
        "image_path": str(p),
        "mask_paths": [],
        "n_masks": 0,
        "y_forged": 0,
    })

# train/forg
missing_train_masks = []
for cid, p in train_forg.items():
    mlist = get_mask_list(str(cid), prefer_train=True)
    if len(mlist) == 0:
        missing_train_masks.append(str(cid))
    rows.append({
        "sample_id": f"{cid}__forg",
        "case_id": str(cid),
        "variant": "forg",
        "image_path": str(p),
        "mask_paths": [str(x) for x in mlist],
        "n_masks": int(len(mlist)),
        "y_forged": 1,
    })

# supplemental
missing_supp_masks = []
for cid, p in supp_map.items():
    mlist = get_mask_list(str(cid), prefer_train=False)
    if len(mlist) == 0:
        missing_supp_masks.append(str(cid))
    rows.append({
        "sample_id": f"{cid}__supp",
        "case_id": str(cid),
        "variant": "supp",
        "image_path": str(p),
        "mask_paths": [str(x) for x in mlist],
        "n_masks": int(len(mlist)),
        "y_forged": 1 if len(mlist) > 0 else 0,
    })

df_train_all = pd.DataFrame(rows)
df_train_all["case_id"] = df_train_all["case_id"].astype(str)

df_train_seg = df_train_all[df_train_all["n_masks"] > 0].reset_index(drop=True)

_p("\nTrain build summary (sample-level):")
_p(f"  df_train_all rows: {len(df_train_all):,}")
_p(str(df_train_all["variant"].value_counts()))
_p("\ny_forged counts:")
_p(str(df_train_all["y_forged"].value_counts().rename(index={0:"authentic",1:"forged"})))
_p(f"\nSegmentation train rows (n_masks>0): {len(df_train_seg):,}")

if len(missing_train_masks) > 0:
    _p(f"\n[WARN] forged image tanpa mask (train_masks tidak ketemu): {len(missing_train_masks):,}")
    _p("  contoh ids: " + ", ".join(missing_train_masks[:20]))
if len(missing_supp_masks) > 0:
    _p(f"\n[WARN] supplemental image tanpa mask (supp_masks tidak ketemu): {len(missing_supp_masks):,}")
    _p("  contoh ids: " + ", ".join(missing_supp_masks[:20]))

# ----------------------------
# df_test aligned to sample_submission order
# ----------------------------
df_sample = pd.read_csv(PATHS["sample_sub"])

# cari kolom case_id & annotation secara case-insensitive
cols_lower = {c.lower(): c for c in df_sample.columns}
if "case_id" not in cols_lower or "annotation" not in cols_lower:
    raise ValueError(f"sample_submission.csv harus punya kolom case_id & annotation. Found: {list(df_sample.columns)}")

df_sample = df_sample.rename(columns={
    cols_lower["case_id"]: "case_id",
    cols_lower["annotation"]: "annotation",
}).copy()
df_sample["case_id"] = df_sample["case_id"].astype(str)

df_test = pd.DataFrame({"case_id": list(test_map.keys())})
df_test["image_path"] = df_test["case_id"].map(lambda x: str(test_map.get(x, "")))
df_test = df_sample[["case_id"]].merge(df_test, on="case_id", how="left")
df_test["image_path"] = df_test["image_path"].fillna("").astype(str)

resolved = int(df_test["image_path"].map(lambda p: Path(p).exists()).sum())
_p(f"\nTest indexed (aligned): {resolved:,}/{len(df_test):,} resolved")
if resolved != len(df_test):
    # tampilkan beberapa yang tidak ketemu
    bad = df_test[df_test["image_path"].map(lambda p: not Path(p).exists())]["case_id"].tolist()
    _p("[WARN] Ada test case_id yang tidak resolve ke file image.")
    _p("  contoh: " + ", ".join(bad[:20]))

# ----------------------------
# RLE utils (order='F' default)
# ----------------------------
RLE_ORDER = "F"

def rle_decode(rle: str, shape_hw: tuple, order: str="F") -> np.ndarray:
    H, W = shape_hw
    s = str(rle).strip()
    if s == "" or s.lower() == "authentic":
        return np.zeros((H, W), dtype=np.uint8)
    nums = np.asarray(list(map(int, s.split())), dtype=np.int64)
    if len(nums) % 2 != 0:
        raise ValueError("Invalid RLE (odd length)")
    starts = nums[0::2] - 1
    lens   = nums[1::2]
    ends = starts + lens
    flat = np.zeros(H*W, dtype=np.uint8)
    for st, en in zip(starts, ends):
        st = max(0, int(st)); en = min(int(en), flat.size)
        if en > st:
            flat[st:en] = 1
    if order.upper() == "F":
        return flat.reshape((W, H)).T
    return flat.reshape((H, W))

def rle_encode(mask: np.ndarray, order: str="F") -> str:
    m = (mask > 0).astype(np.uint8)
    if m.sum() == 0:
        return ""
    pixels = m.T.reshape(-1) if order.upper() == "F" else m.reshape(-1)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(map(str, runs))

# ----------------------------
# Sanity: cek mask align + RLE roundtrip pada beberapa sample seg
# ----------------------------
pool = df_train_seg.copy()
sample_n = min(20, len(pool))
sample_rows = pool.sample(sample_n, random_state=2025) if sample_n > 0 else pool

shape_mismatch = 0
roundtrip_ok = 0
nonempty = 0
comp_stats = []
area_fracs = []

for _, row in sample_rows.iterrows():
    ip = row["image_path"]
    mpaths = row["mask_paths"]
    if (not Path(ip).exists()) or (len(mpaths) == 0):
        continue
    with Image.open(ip) as im:
        W, H = im.size
    um = union_masks_aligned(mpaths, H, W)
    if um is None:
        continue
    if um.shape != (H, W):
        shape_mismatch += 1
    s = int(um.sum())
    if s > 0:
        nonempty += 1
        area_fracs.append(s / float(H*W))
    rle = rle_encode(um, order=RLE_ORDER)
    um2 = rle_decode(rle, (H, W), order=RLE_ORDER)
    if np.array_equal(um, um2):
        roundtrip_ok += 1
    if _HAS_SCIPY:
        _, n = ndi.label(um.astype(bool))
        comp_stats.append(int(n))

_p("\nSanity (seg samples):")
_p(f"  sampled: {sample_n}")
_p(f"  mask/image shape mismatch: {shape_mismatch}")
_p(f"  non-empty union masks: {nonempty}/{sample_n}")
_p(f"  RLE roundtrip ok: {roundtrip_ok}/{sample_n} (RLE_ORDER='{RLE_ORDER}')")
if len(area_fracs):
    _p(f"  area_frac (min/med/max): {min(area_fracs):.6f}/{float(np.median(area_fracs)):.6f}/{max(area_fracs):.6f}")
if _HAS_SCIPY and len(comp_stats):
    _p(f"  components (min/med/max): {min(comp_stats)}/{int(np.median(comp_stats))}/{max(comp_stats)}")

# ----------------------------
# CV: fold per GROUP case_id (anti leakage)
# Stratify: has_auth (distribusi negatif merata)
# ----------------------------
N_FOLDS = 5
SEED = 2025

grp = df_train_all.groupby("case_id").agg(
    has_auth=("variant", lambda x: int("auth" in set(x))),
    n_samples=("sample_id", "count"),
).reset_index()

if len(grp) < N_FOLDS:
    raise ValueError(f"Jumlah group case_id ({len(grp)}) < N_FOLDS ({N_FOLDS}). Turunkan N_FOLDS.")

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
grp["fold"] = -1
y_strat = grp["has_auth"].values.astype(int)

for f, (_, va) in enumerate(skf.split(np.zeros(len(grp)), y_strat)):
    grp.loc[va, "fold"] = f

fold_map = dict(zip(grp["case_id"].astype(str), grp["fold"].astype(int)))
df_train_all["fold"] = df_train_all["case_id"].map(fold_map).astype(int)
df_train_seg["fold"] = df_train_seg["case_id"].map(fold_map).astype(int)

_p(f"\nCV ready (grouped by case_id): n_splits={N_FOLDS}")
_p("fold has_auth_rate:")
_p(grp.groupby("fold")["has_auth"].mean().to_string())
_p("\nfold sample y_forged rate (df_train_all):")
_p(df_train_all.groupby("fold")["y_forged"].mean().to_string())

_p("\nTrain_all head:")
_p(str(df_train_all.head()))
_p("\nTrain_seg head:")
_p(str(df_train_seg.head()))
_p("\nTest head:")
_p(str(df_test.head()))

_p("\nDONE. Exported objects:")
_p("- DATA_ROOT, PATHS, DINO_BASE_DIR")
_p("- df_train_all (sample-level), df_train_seg (mask-only)")
_p("- df_test aligned to sample_submission")
_p("- RLE_ORDER, rle_encode, rle_decode")
_p("- load_mask_any, union_masks_aligned")


OK PATHS:
     data_root: /kaggle/input/recodai-luc-scientific-image-forgery-detection
   supp_images: /kaggle/input/recodai-luc-scientific-image-forgery-detection/supplemental_images
    supp_masks: /kaggle/input/recodai-luc-scientific-image-forgery-detection/supplemental_masks
   test_images: /kaggle/input/recodai-luc-scientific-image-forgery-detection/test_images
  train_images: /kaggle/input/recodai-luc-scientific-image-forgery-detection/train_images
   train_masks: /kaggle/input/recodai-luc-scientific-image-forgery-detection/train_masks
    sample_sub: /kaggle/input/recodai-luc-scientific-image-forgery-detection/sample_submission.csv
     dino_base: /kaggle/input/dinov2/pytorch/base/1
  scipy: True

[/kaggle/input/recodai-luc-scientific-image-forgery-detection/train_images] files=5,128 ext_counts={'.png': 5128}
  - forged/50028.png
  - forged/18054.png
  - forged/32154.png
  - forged/51742.png
  - forged/60154.png
  - forged/22739.png

[/kaggle/input/recodai-luc-scientific-image-f

# DINOv2-Base Feature Cache (CPU-Optimized)

In [ ]:
# ============================================================
# STAGE 2 — DINOv2-Base Feature Cache (CPU-Optimized) (ONE CELL, REVISI FULL v3)
# - Multi-scale (BASE+HI), fuse (concat/avg), light whitening
# - CACHE VERSIONING: cfg_id + auto-rebuild jika config berubah
# - Model load robust (trust_remote_code fallback)
# - Resize snap CEIL ke multiple patch (lebih aman untuk detail)
# Output per item (tetap kompatibel STAGE 3):
# - patch_desc: (H_p, W_p, D_fused) float16 (L2-normalized)
# - cls_desc  : (D_fused,) float16 (L2-normalized)
# - meta: JSON string
# ============================================================

import os, json, math, gc, time, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

import torch
import torch.nn.functional as F
from transformers import AutoModel

# allow truncated image (kadang ada file aneh, ini mencegah crash)
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ----------------------------
# Guards
# ----------------------------
need_cols = {"sample_id","case_id","image_path","variant","y_forged","fold"}
if "df_train_all" not in globals() or not isinstance(df_train_all, pd.DataFrame):
    raise RuntimeError("df_train_all belum ada. Jalankan STAGE 1 dulu.")
if not need_cols.issubset(set(df_train_all.columns)):
    raise RuntimeError(f"df_train_all missing cols {need_cols - set(df_train_all.columns)}")

if "df_test" not in globals() or not isinstance(df_test, pd.DataFrame):
    raise RuntimeError("df_test belum ada. Jalankan STAGE 1 dulu.")
if not {"case_id","image_path"}.issubset(set(df_test.columns)):
    raise RuntimeError("df_test harus punya kolom case_id, image_path.")

if "DINO_BASE_DIR" not in globals():
    raise RuntimeError("DINO_BASE_DIR belum ada. Pastikan STAGE 1 mendefinisikannya.")
DINO_BASE_DIR = Path(str(DINO_BASE_DIR))
if not DINO_BASE_DIR.exists():
    raise FileNotFoundError(f"DINO_BASE_DIR tidak ditemukan: {DINO_BASE_DIR}")

# ----------------------------
# Config (CPU-friendly)
# ----------------------------
PATCH_SIZE = 14

USE_MULTI_SCALE = True
MAX_SIDE_BASE   = 384
MAX_SIDE_HI     = 512
MIN_SIDE        = 224

FUSE_MODE = "concat"          # "concat" atau "avg"
USE_LIGHT_WHITEN = True
WHITEN_EPS = 1e-6

USE_FP16_STORE = True
N_LIMIT_TRAIN  = None
N_LIMIT_TEST   = None

# threads (CPU) - set konservatif biar stabil
try:
    torch.set_num_threads(max(1, (os.cpu_count() or 2)//2))
    torch.set_num_interop_threads(1)
except Exception:
    pass

device = torch.device("cpu")

# ----------------------------
# Cache versioning (anti cache salah / config berubah)
# ----------------------------
CFG = {
    "patch_size": PATCH_SIZE,
    "use_multi_scale": USE_MULTI_SCALE,
    "max_side_base": MAX_SIDE_BASE,
    "max_side_hi": (MAX_SIDE_HI if USE_MULTI_SCALE else MAX_SIDE_BASE),
    "min_side": MIN_SIDE,
    "fuse_mode": FUSE_MODE,
    "use_light_whiten": USE_LIGHT_WHITEN,
    "whiten_eps": WHITEN_EPS,
    "use_fp16_store": USE_FP16_STORE,
    "dino_base_dir": str(DINO_BASE_DIR),
}
_cfg_blob = json.dumps(CFG, sort_keys=True).encode("utf-8")
CFG_ID = hashlib.sha1(_cfg_blob).hexdigest()[:12]  # short id, cukup untuk folder

CACHE_ROOT  = Path("/kaggle/working/recodai_luc/cache/dino_v2_base") / f"cfg_{CFG_ID}"
CACHE_TRAIN = CACHE_ROOT / "train_all"
CACHE_TEST  = CACHE_ROOT / "test"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_TRAIN.mkdir(parents=True, exist_ok=True)
CACHE_TEST.mkdir(parents=True, exist_ok=True)

print("CFG_ID:", CFG_ID)
print("CACHE_ROOT:", CACHE_ROOT)

# ----------------------------
# Normalization (ImageNet)
# ----------------------------
IMNET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMNET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def _ceil_to_multiple(x, m):
    return int(max(m, math.ceil(float(x)/m) * m))

def resize_keep_ar(img: Image.Image, max_side: int, min_side: int, patch: int):
    """
    Resize jaga aspect ratio.
    Long side -> clamp [min_side, max_side], lalu snap H,W ke multiple patch (CEIL).
    """
    W0, H0 = img.size
    long0 = max(W0, H0)
    if long0 <= 0:
        return img, (H0, W0), (H0, W0)

    target_long = min(max_side, max(min_side, long0))
    scale = target_long / float(long0)

    W1 = max(patch, int(round(W0 * scale)))
    H1 = max(patch, int(round(H0 * scale)))

    W1 = _ceil_to_multiple(W1, patch)
    H1 = _ceil_to_multiple(H1, patch)

    if (W1, H1) != (W0, H0):
        img = img.resize((W1, H1), resample=Image.BICUBIC)

    return img, (H0, W0), (H1, W1)

def pil_to_tensor_norm(img: Image.Image):
    arr = np.array(img.convert("RGB"), dtype=np.float32) / 255.0
    arr = (arr - IMNET_MEAN) / IMNET_STD
    t = torch.from_numpy(arr).permute(2,0,1).unsqueeze(0).contiguous()
    return t

def l2_normalize(x: torch.Tensor, dim=-1, eps=1e-12):
    return x / (x.norm(dim=dim, keepdim=True) + eps)

def _infer_grid_from_tokens(N: int, H1: int, W1: int, patch: int):
    gh = max(1, H1 // patch)
    gw = max(1, W1 // patch)
    if gh * gw == N:
        return gh, gw

    # fallback factorization dekat aspect ratio
    ar = W1 / max(1, H1)
    gw0 = int(round(math.sqrt(N * ar)))
    gw0 = max(1, min(gw0, N))
    gh0 = max(1, N // gw0)
    gw0 = max(1, N // gh0)
    if gh0 * gw0 != N:
        return 1, N
    return gh0, gw0

@torch.inference_mode()
def _forward_dino(img_pil: Image.Image):
    x = pil_to_tensor_norm(img_pil).to(device, dtype=torch.float32)
    y = model(pixel_values=x).last_hidden_state  # [1, 1+N, D]
    cls = y[:, 0, :]    # [1,D]
    pt  = y[:, 1:, :]   # [1,N,D]
    return cls, pt

def _tokens_to_grid(pt: torch.Tensor, H1: int, W1: int, patch: int):
    N = int(pt.shape[1])
    D = int(pt.shape[2])
    gh, gw = _infer_grid_from_tokens(N, H1, W1, patch)
    pt0 = pt.squeeze(0)  # [N,D]
    pt0 = l2_normalize(pt0, dim=-1)
    try:
        grid = pt0.reshape(gh, gw, D).contiguous()
    except Exception:
        grid = pt0.reshape(1, N, D).contiguous()
        gh, gw = 1, N
    return grid, gh, gw, D

def _resample_grid_to(grid: torch.Tensor, gh_t: int, gw_t: int):
    gh, gw, D = grid.shape
    x = grid.permute(2,0,1).unsqueeze(0).contiguous()  # [1,D,gh,gw]
    x2 = F.interpolate(x, size=(gh_t, gw_t), mode="bilinear", align_corners=False)
    g2 = x2.squeeze(0).permute(1,2,0).contiguous()     # [gh_t,gw_t,D]
    return g2

def _light_whiten_patches(grid: torch.Tensor, eps: float = 1e-6):
    gh, gw, D = grid.shape
    x = grid.reshape(-1, D)
    mu = x.mean(dim=0, keepdim=True)
    sd = x.std(dim=0, keepdim=True).clamp_min(eps)
    xw = (x - mu) / sd
    xw = l2_normalize(xw, dim=-1)
    return xw.reshape(gh, gw, D).contiguous()

# ----------------------------
# Load DINOv2 Base (local, robust)
# ----------------------------
print("\nLoading DINOv2-Base from:", DINO_BASE_DIR)
model = None
load_err = None
for kwargs in (
    {"local_files_only": True, "trust_remote_code": True},
    {"local_files_only": True},
):
    try:
        model = AutoModel.from_pretrained(str(DINO_BASE_DIR), **kwargs)
        load_err = None
        break
    except Exception as e:
        load_err = e

if model is None:
    raise RuntimeError(f"Gagal load DINOv2 dari {DINO_BASE_DIR}. Last err: {type(load_err).__name__}: {load_err}")

model.eval().to(device)

with torch.inference_mode():
    dummy = torch.zeros((1,3,224,224), dtype=torch.float32, device=device)
    out = model(pixel_values=dummy)
    D0 = int(out.last_hidden_state.shape[-1])
print("Model loaded. Base embed dim:", D0, "| device:", device)

def extract_features_one(image_path: str):
    """
    Return:
    - patch_desc: (H_p, W_p, D_fused) float32 (L2 norm)
    - cls_desc  : (D_fused,) float32 (L2 norm)
    - meta dict
    """
    with Image.open(image_path) as im:
        img0 = im.convert("RGB")

    # BASE
    img_b, (H0,W0), (Hb,Wb) = resize_keep_ar(img0, max_side=MAX_SIDE_BASE, min_side=MIN_SIDE, patch=PATCH_SIZE)
    cls_b, pt_b = _forward_dino(img_b)
    cls_b = l2_normalize(cls_b, dim=-1).squeeze(0)
    grid_b, gh, gw, D = _tokens_to_grid(pt_b, Hb, Wb, PATCH_SIZE)

    if USE_MULTI_SCALE:
        img_h, _, (Hh,Wh) = resize_keep_ar(img0, max_side=MAX_SIDE_HI, min_side=MIN_SIDE, patch=PATCH_SIZE)
        cls_h, pt_h = _forward_dino(img_h)
        cls_h = l2_normalize(cls_h, dim=-1).squeeze(0)
        grid_h, gh2, gw2, _ = _tokens_to_grid(pt_h, Hh, Wh, PATCH_SIZE)

        grid_h_rs = _resample_grid_to(grid_h, gh, gw)
        grid_h_rs = l2_normalize(grid_h_rs.reshape(-1, D), dim=-1).reshape(gh, gw, D).contiguous()

        if FUSE_MODE == "avg":
            grid_f = l2_normalize((grid_b + grid_h_rs), dim=-1)
            cls_f  = l2_normalize((cls_b + cls_h), dim=-1)
            D_fused = D
        else:
            grid_f = torch.cat([grid_b, grid_h_rs], dim=-1)
            grid_f = l2_normalize(grid_f.reshape(-1, grid_f.shape[-1]), dim=-1).reshape(gh, gw, -1).contiguous()
            cls_f  = torch.cat([cls_b, cls_h], dim=-1)
            cls_f  = l2_normalize(cls_f, dim=-1)
            D_fused = int(grid_f.shape[-1])
    else:
        grid_f = grid_b
        cls_f  = cls_b
        D_fused = D

    if USE_LIGHT_WHITEN:
        grid_f = _light_whiten_patches(grid_f, eps=WHITEN_EPS)

    meta = {
        "cfg_id": CFG_ID,
        "orig_hw": [int(H0), int(W0)],
        "resized_hw_base": [int(Hb), int(Wb)],
        "patch_size": int(PATCH_SIZE),
        "grid_hw": [int(gh), int(gw)],
        "embed_dim_base": int(D0),
        "embed_dim_fused": int(D_fused),
        "max_side_base": int(MAX_SIDE_BASE),
        "max_side_hi": int(MAX_SIDE_HI) if USE_MULTI_SCALE else int(MAX_SIDE_BASE),
        "min_side": int(MIN_SIDE),
        "use_multi_scale": bool(USE_MULTI_SCALE),
        "fuse_mode": str(FUSE_MODE),
        "use_light_whiten": bool(USE_LIGHT_WHITEN),
    }
    return grid_f, cls_f, meta

def save_npz(path: Path, patch_desc: torch.Tensor, cls_desc: torch.Tensor, meta: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    pd_np = patch_desc.cpu().numpy()
    cd_np = cls_desc.cpu().numpy()
    if USE_FP16_STORE:
        pd_np = pd_np.astype(np.float16)
        cd_np = cd_np.astype(np.float16)
    else:
        pd_np = pd_np.astype(np.float32)
        cd_np = cd_np.astype(np.float32)

    np.savez(
        str(path),
        patch_desc=pd_np,
        cls_desc=cd_np,
        meta=json.dumps(meta),
    )

def read_cfg_id_from_npz(npz_path: Path):
    try:
        z = np.load(str(npz_path), allow_pickle=False)
        meta_s = z["meta"].item() if hasattr(z["meta"], "item") else z["meta"]
        meta = json.loads(str(meta_s))
        return str(meta.get("cfg_id", ""))
    except Exception:
        return ""

def cache_loop(df: pd.DataFrame, id_col: str, out_dir: Path, limit=None, label="train"):
    n = len(df) if limit is None else min(len(df), int(limit))
    miss = 0
    done = 0
    rebuilt = 0
    t0 = time.time()

    # itertuples lebih cepat dari iloc loop
    rows = df.head(n).itertuples(index=False)

    for i, r in enumerate(rows, start=1):
        uid = str(getattr(r, id_col))
        ip  = str(getattr(r, "image_path"))
        out = out_dir / f"{uid}.npz"

        if out.exists():
            old = read_cfg_id_from_npz(out)
            if old == CFG_ID:
                done += 1
                continue
            else:
                # config berubah atau meta rusak -> rebuild
                try:
                    out.unlink()
                except Exception:
                    pass
                rebuilt += 1

        if (not ip) or (not Path(ip).exists()):
            miss += 1
            continue

        try:
            pt, cls, meta = extract_features_one(ip)
            save_npz(out, pt, cls, meta)
            done += 1
        except Exception as e:
            print(f"[{label}] FAIL uid={uid} path={ip} err={type(e).__name__}: {e}")
            miss += 1

        if (i % 50) == 0:
            dt = time.time() - t0
            rate = i / max(dt, 1e-9)
            print(f"[{label}] {i}/{n} | cached_ok={done} | rebuilt={rebuilt} | miss/fail={miss} | {rate:.2f} it/s | elapsed={dt:.1f}s")
            gc.collect()

    dt = time.time() - t0
    print(f"\n[{label}] DONE. total={n} cached_ok={done} rebuilt={rebuilt} miss/fail={miss} elapsed={dt:.1f}s")
    return {"label": label, "total": int(n), "cached_ok": int(done), "rebuilt": int(rebuilt), "miss_fail": int(miss), "elapsed_s": float(dt)}

# ----------------------------
# Run cache
# ----------------------------
df_train_run = df_train_all.reset_index(drop=True).copy()
df_test_run  = df_test.reset_index(drop=True).copy()

df_train_run = df_train_run.sort_values(["variant","fold","case_id","sample_id"]).reset_index(drop=True)
df_test_run  = df_test_run.sort_values(["case_id"]).reset_index(drop=True)

print("\nCaching TRAIN_ALL features...")
rep_train = cache_loop(df_train_run, id_col="sample_id", out_dir=CACHE_TRAIN, limit=N_LIMIT_TRAIN, label="train_all")

print("\nCaching TEST features...")
rep_test = cache_loop(df_test_run, id_col="case_id", out_dir=CACHE_TEST, limit=N_LIMIT_TEST, label="test")

# ----------------------------
# Write manifests
# ----------------------------
def build_manifest(df: pd.DataFrame, id_col: str, out_dir: Path):
    recs = []
    for r in df.itertuples(index=False):
        uid = str(getattr(r, id_col))
        case_id = str(getattr(r, "case_id", ""))
        variant = str(getattr(r, "variant", ""))
        fold = int(getattr(r, "fold", -1)) if hasattr(r, "fold") else -1
        y_forged = int(getattr(r, "y_forged", -1)) if hasattr(r, "y_forged") else -1
        ip = str(getattr(r, "image_path", ""))

        p = out_dir / f"{uid}.npz"
        recs.append({
            id_col: uid,
            "case_id": case_id,
            "variant": variant,
            "fold": fold,
            "y_forged": y_forged,
            "image_path": ip,
            "feat_path": str(p),
            "feat_exists": int(p.exists()),
        })
    return pd.DataFrame(recs)

man_train = build_manifest(df_train_run, "sample_id", CACHE_TRAIN)
man_test  = build_manifest(df_test_run,  "case_id",  CACHE_TEST)

MAN_TRAIN_PATH = CACHE_ROOT / "manifest_train_all.csv"
MAN_TEST_PATH  = CACHE_ROOT / "manifest_test.csv"
man_train.to_csv(MAN_TRAIN_PATH, index=False)
man_test.to_csv(MAN_TEST_PATH, index=False)

summary = {
    "cfg": CFG,
    "cfg_id": CFG_ID,
    "cache_root": str(CACHE_ROOT),
    "train": rep_train,
    "test": rep_test,
}
with open(CACHE_ROOT / "cache_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nWrote manifests:")
print(" -", MAN_TRAIN_PATH)
print(" -", MAN_TEST_PATH)
print(" -", CACHE_ROOT / "cache_summary.json")

print("\nDONE. Exported objects:")
print("- model (DINOv2-base), CACHE_ROOT, MAN_TRAIN_PATH, MAN_TEST_PATH, CFG_ID")
print("- helper: extract_features_one()")


CFG_ID: 2e9b2143c2d3
CACHE_ROOT: /kaggle/working/recodai_luc/cache/dino_v2_base/cfg_2e9b2143c2d3

Loading DINOv2-Base from: /kaggle/input/dinov2/pytorch/base/1


2026-01-01 13:45:56.381236: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767275156.760136      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767275156.871868      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767275157.829383      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767275157.829437      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767275157.829441      55 computation_placer.cc:177] computation placer alr

Model loaded. Base embed dim: 768 | device: cpu

Caching TRAIN_ALL features...
[train_all] 50/5176 | cached_ok=50 | rebuilt=0 | miss/fail=0 | 0.39 it/s | elapsed=127.6s
[train_all] 100/5176 | cached_ok=100 | rebuilt=0 | miss/fail=0 | 0.38 it/s | elapsed=260.9s
[train_all] 150/5176 | cached_ok=150 | rebuilt=0 | miss/fail=0 | 0.39 it/s | elapsed=386.9s
[train_all] 200/5176 | cached_ok=200 | rebuilt=0 | miss/fail=0 | 0.39 it/s | elapsed=518.4s
[train_all] 250/5176 | cached_ok=250 | rebuilt=0 | miss/fail=0 | 0.39 it/s | elapsed=646.9s
[train_all] 300/5176 | cached_ok=300 | rebuilt=0 | miss/fail=0 | 0.38 it/s | elapsed=790.9s
[train_all] 350/5176 | cached_ok=350 | rebuilt=0 | miss/fail=0 | 0.38 it/s | elapsed=915.5s
[train_all] 400/5176 | cached_ok=400 | rebuilt=0 | miss/fail=0 | 0.38 it/s | elapsed=1051.8s
[train_all] 450/5176 | cached_ok=450 | rebuilt=0 | miss/fail=0 | 0.37 it/s | elapsed=1204.2s
[train_all] 500/5176 | cached_ok=500 | rebuilt=0 | miss/fail=0 | 0.37 it/s | elapsed=1346.2s


# Robust Matching (Top-k + MNN + Multi-Peak Translation)

In [ ]:
# ============================================================
# STAGE 3 — Robust Matching (Top-k + TRUE MNN + Ratio/Margin + Peak NMS) (ONE CELL, REVISI FULL v5)
# - Kompatibel STAGE 4: best_src, best_dst, best_sim + scalars tetap ada.
# - Upgrade: auto-find manifest, match cache versioning, ratio/margin fix, TRUE MNN vectorized, loop lebih cepat.
# ============================================================

import os, gc, time, json, math, hashlib
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

# ----------------------------
# REQUIRE (sesuai STAGE 1)
# ----------------------------
for need in ["df_train_all", "df_test"]:
    if need not in globals():
        raise RuntimeError(f"Missing: {need}. Jalankan STAGE 1 dulu.")

df_train_all = df_train_all.copy()
df_test = df_test.copy()

for c in ["sample_id", "case_id"]:
    if c in df_train_all.columns:
        df_train_all[c] = df_train_all[c].astype(str)
if "case_id" in df_test.columns:
    df_test["case_id"] = df_test["case_id"].astype(str)
if "image_path" not in df_test.columns:
    df_test["image_path"] = ""

# ----------------------------
# Auto-find manifests from STAGE 2 (support cfg_* folder)
# ----------------------------
def _auto_find_manifest(default_path: Path, pattern_name: str) -> Path:
    if default_path.exists():
        return default_path
    base = Path("/kaggle/working/recodai_luc/cache/dino_v2_base")
    if not base.exists():
        return default_path
    cands = list(base.rglob(pattern_name))
    if not cands:
        return default_path
    # pilih yang paling baru (mtime)
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]

if "MAN_TRAIN_PATH" in globals():
    MAN_TRAIN_PATH = Path(str(MAN_TRAIN_PATH))
else:
    MAN_TRAIN_PATH = _auto_find_manifest(
        Path("/kaggle/working/recodai_luc/cache/dino_v2_base/manifest_train_all.csv"),
        "manifest_train_all.csv"
    )

if "MAN_TEST_PATH" in globals():
    MAN_TEST_PATH = Path(str(MAN_TEST_PATH))
else:
    MAN_TEST_PATH = _auto_find_manifest(
        Path("/kaggle/working/recodai_luc/cache/dino_v2_base/manifest_test.csv"),
        "manifest_test.csv"
    )

if not MAN_TRAIN_PATH.exists():
    raise FileNotFoundError(f"Manifest train tidak ditemukan: {MAN_TRAIN_PATH} (jalankan STAGE 2 dulu).")
if not MAN_TEST_PATH.exists():
    raise FileNotFoundError(f"Manifest test tidak ditemukan: {MAN_TEST_PATH} (jalankan STAGE 2 dulu).")

man_train = pd.read_csv(MAN_TRAIN_PATH, dtype=str, low_memory=False)
man_test  = pd.read_csv(MAN_TEST_PATH,  dtype=str, low_memory=False)

need_train_cols = {"sample_id","feat_path","feat_exists"}
need_test_cols  = {"case_id","feat_path","feat_exists"}
if not need_train_cols.issubset(set(man_train.columns)):
    raise RuntimeError(f"manifest_train_all.csv missing cols: {need_train_cols - set(man_train.columns)}")
if not need_test_cols.issubset(set(man_test.columns)):
    raise RuntimeError(f"manifest_test.csv missing cols: {need_test_cols - set(man_test.columns)}")

man_train["feat_exists"] = man_train["feat_exists"].astype(int)
man_test["feat_exists"]  = man_test["feat_exists"].astype(int)

man_train = man_train[man_train["feat_exists"] == 1].copy()
man_test  = man_test[man_test["feat_exists"]  == 1].copy()

print("MAN_TRAIN_PATH:", MAN_TRAIN_PATH, "| rows(feat_exists=1):", len(man_train))
print("MAN_TEST_PATH :", MAN_TEST_PATH,  "| rows(feat_exists=1):", len(man_test))

# ----------------------------
# Output dirs (versioned)
# ----------------------------
CFG_MATCH = {
    "topk": 80,
    "sim_thr": 0.78,
    "min_sep": 4,
    "close_metric": "max",   # "max" / "both"
    "ratio_thr": 1.05,
    "margin_thr": 0.02,
    "bin_step": 1,
    "peaks_M": 3,
    "min_peak_count": 20,
    "peak_min_sep": 2,
    "weight_power": 2.0,
    "margin_power": 1.0,
    "skip_if_exists": True,
    "print_every": 200,
    "limit_train": None,
    "limit_test": None,
}

# cfg id supaya match cache tidak ketiban dan tidak pakai versi salah
_match_blob = json.dumps(CFG_MATCH, sort_keys=True).encode("utf-8")
MATCH_CFG_ID = hashlib.sha1(_match_blob).hexdigest()[:12]

MATCH_ROOT = Path("/kaggle/working/recodai_luc/cache") / f"match_base_v3_cfg_{MATCH_CFG_ID}"
MATCH_TRAIN_DIR = MATCH_ROOT / "train_all"
MATCH_TEST_DIR  = MATCH_ROOT / "test"
MATCH_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
MATCH_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("MATCH_CFG_ID  :", MATCH_CFG_ID)
print("MATCH_ROOT    :", MATCH_ROOT)

try:
    torch.set_num_threads(max(1, (os.cpu_count() or 2)//2))
except Exception:
    pass

# ----------------------------
# Helpers: load DINO cache (STAGE 2 format)
# ----------------------------
def load_dino_patch_from_npz(npz_path: str):
    p = Path(npz_path)
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)

    if "patch_desc" in z.files:
        pdsc = z["patch_desc"]
        if pdsc.ndim != 3:
            return None
        gh, gw, D = pdsc.shape
        feat = pdsc.reshape(gh*gw, D).astype(np.float32, copy=False)
        return feat, int(gh), int(gw)

    if "feat" in z.files and "grid_h" in z.files and "grid_w" in z.files:
        feat = z["feat"].astype(np.float32, copy=False)
        gh = int(z["grid_h"]); gw = int(z["grid_w"])
        return feat, gh, gw

    return None

def save_match_npz(out_path: Path, payload: dict):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(str(out_path), **payload)

def read_match_cfg_id(npz_path: Path):
    try:
        z = np.load(str(npz_path), allow_pickle=False)
        if "match_cfg_id" in z.files:
            v = z["match_cfg_id"]
            # v bisa scalar array
            return str(v.item()) if hasattr(v, "item") else str(v)
    except Exception:
        pass
    return ""

@lru_cache(maxsize=256)
def _grid_rc_np(gh: int, gw: int):
    rr = np.repeat(np.arange(gh, dtype=np.int32), gw)
    cc = np.tile(np.arange(gw, dtype=np.int32), gh)
    return rr, cc

def _is_close(src_idx: np.ndarray, dst_idx: np.ndarray, gh: int, gw: int, min_sep: int, metric: str):
    rr, cc = _grid_rc_np(gh, gw)
    dr = np.abs(rr[src_idx] - rr[dst_idx])
    dc = np.abs(cc[src_idx] - cc[dst_idx])
    if metric == "both":
        return (dr < min_sep) & (dc < min_sep)
    return (np.maximum(dr, dc) < min_sep)

def canonicalize_offsets(dy: np.ndarray, dx: np.ndarray):
    dy = dy.astype(np.int32, copy=False)
    dx = dx.astype(np.int32, copy=False)
    neg = (dy < 0) | ((dy == 0) & (dx < 0))
    dy2 = dy.copy(); dx2 = dx.copy()
    dy2[neg] = -dy2[neg]
    dx2[neg] = -dx2[neg]
    return dy2, dx2

def _encode_key(dy: np.ndarray, dx: np.ndarray):
    return (dy.astype(np.int64) << 32) ^ (dx.astype(np.int64) & np.int64(0xffffffff))

def robust_match_one(feat_np: np.ndarray, gh: int, gw: int, cfg: dict):
    N = int(gh * gw)
    if feat_np is None or feat_np.ndim != 2 or feat_np.shape[0] != N:
        return {
            "grid_h": np.int32(gh), "grid_w": np.int32(gw),
            "has_peak": np.int8(0), "peak_ratio": np.float32(0.0),
            "peaks_dy_dx": np.zeros((0,2), dtype=np.int16),
            "peaks_weight": np.zeros((0,), dtype=np.float32),
            "peaks_count": np.zeros((0,), dtype=np.int32),
            "best_src": np.zeros((0,), dtype=np.int32),
            "best_dst": np.zeros((0,), dtype=np.int32),
            "best_sim": np.zeros((0,), dtype=np.float16),
            "best_weight": np.float32(0.0),
            "best_count": np.int32(0),
            "best_mean_sim": np.float32(0.0),
            "n_pairs_thr": np.int32(0),
            "n_pairs_mnn": np.int32(0),
            "best_inlier_ratio": np.float32(0.0),
            "best_weight_frac": np.float32(0.0),
        }

    topk = int(cfg["topk"])
    sim_thr = float(cfg["sim_thr"])
    min_sep = int(cfg["min_sep"])
    metric  = str(cfg.get("close_metric", "max"))
    ratio_thr  = float(cfg["ratio_thr"])
    margin_thr = float(cfg["margin_thr"])

    bin_step = int(cfg["bin_step"])
    peaks_M = int(cfg["peaks_M"])
    min_peak_count = int(cfg["min_peak_count"])
    peak_min_sep = int(cfg["peak_min_sep"])
    wpow = float(cfg["weight_power"])
    mpow = float(cfg["margin_power"])

    # normalize features
    f = torch.from_numpy(feat_np).to(torch.float32)
    f = F.normalize(f, dim=1)

    # sim matrix (N,N) - N kecil (grid), aman; robust & simpel
    sim = f @ f.T
    sim.fill_diagonal_(-1e9)

    k = min(max(2, topk), N-1)
    vals, idxs = torch.topk(sim, k=k, dim=1, largest=True, sorted=True)  # [N,k]

    # candidate validity: far + sim>=thr
    src_idx = torch.arange(N, dtype=torch.int64).unsqueeze(1).expand(N, k)
    src_np = src_idx.reshape(-1).cpu().numpy().astype(np.int32, copy=False)
    dst_np = idxs.reshape(-1).cpu().numpy().astype(np.int32, copy=False)

    close_np = _is_close(src_np, dst_np, gh, gw, min_sep=min_sep, metric=metric)
    close = torch.from_numpy(close_np.reshape(N, k))

    valid = (~close) & (vals >= sim_thr)
    has_any = valid.any(dim=1)

    vals_valid = vals.masked_fill(~valid, -1e9)
    best_j = torch.argmax(vals_valid, dim=1)
    chosen_sim = vals_valid.gather(1, best_j.unsqueeze(1)).squeeze(1)
    chosen_dst = idxs.gather(1, best_j.unsqueeze(1)).squeeze(1)

    # top2 valid untuk ratio/margin
    top2v = torch.topk(vals_valid, k=min(2, k), dim=1, largest=True, sorted=True).values
    v1 = top2v[:, 0]
    v2 = top2v[:, 1] if top2v.shape[1] > 1 else torch.full_like(v1, -1e9)

    # FIX penting: jika hanya 1 kandidat valid (v2 ~ -1e9), treat distinctive
    only_one = v2 < -1e8
    eps = 1e-9
    ratio = torch.where(only_one, torch.full_like(v1, 1e9), v1 / (v2 + eps))
    margin = torch.where(only_one, torch.full_like(v1, 1e9), v1 - v2)

    keep = has_any & (v1 > -1e8) & (ratio >= ratio_thr) & (margin >= margin_thr)

    n_pairs_thr = int((vals >= sim_thr).sum().item())
    if keep.sum().item() == 0:
        return {
            "grid_h": np.int32(gh), "grid_w": np.int32(gw),
            "has_peak": np.int8(0), "peak_ratio": np.float32(0.0),
            "peaks_dy_dx": np.zeros((0,2), dtype=np.int16),
            "peaks_weight": np.zeros((0,), dtype=np.float32),
            "peaks_count": np.zeros((0,), dtype=np.int32),
            "best_src": np.zeros((0,), dtype=np.int32),
            "best_dst": np.zeros((0,), dtype=np.int32),
            "best_sim": np.zeros((0,), dtype=np.float16),
            "best_weight": np.float32(0.0),
            "best_count": np.int32(0),
            "best_mean_sim": np.float32(0.0),
            "n_pairs_thr": np.int32(n_pairs_thr),
            "n_pairs_mnn": np.int32(0),
            "best_inlier_ratio": np.float32(0.0),
            "best_weight_frac": np.float32(0.0),
        }

    # 1NN pairs after filter
    src_keep = torch.nonzero(keep, as_tuple=False).squeeze(1).cpu().numpy().astype(np.int32)
    dst_keep = chosen_dst[keep].cpu().numpy().astype(np.int32)
    sim_keep = chosen_sim[keep].cpu().numpy().astype(np.float32)
    mar_keep = margin[keep].cpu().numpy().astype(np.float32)

    # TRUE MNN (vectorized): pilih src terbaik per dst berdasarkan sim_keep
    P = len(src_keep)
    if P == 0:
        return {
            "grid_h": np.int32(gh), "grid_w": np.int32(gw),
            "has_peak": np.int8(0), "peak_ratio": np.float32(0.0),
            "peaks_dy_dx": np.zeros((0,2), dtype=np.int16),
            "peaks_weight": np.zeros((0,), dtype=np.float32),
            "peaks_count": np.zeros((0,), dtype=np.int32),
            "best_src": np.zeros((0,), dtype=np.int32),
            "best_dst": np.zeros((0,), dtype=np.int32),
            "best_sim": np.zeros((0,), dtype=np.float16),
            "best_weight": np.float32(0.0),
            "best_count": np.int32(0),
            "best_mean_sim": np.float32(0.0),
            "n_pairs_thr": np.int32(n_pairs_thr),
            "n_pairs_mnn": np.int32(0),
            "best_inlier_ratio": np.float32(0.0),
            "best_weight_frac": np.float32(0.0),
        }

    # sort by (dst asc, sim desc)
    order = np.lexsort((-sim_keep, dst_keep))
    dst_s = dst_keep[order]
    src_s = src_keep[order]
    sim_s = sim_keep[order]
    mar_s = mar_keep[order]

    # first index per dst = best sim (karena sim desc di dalam dst group)
    first = np.ones(P, dtype=bool)
    first[1:] = (dst_s[1:] != dst_s[:-1])
    best_dst = dst_s[first]
    best_src = src_s[first]

    dst_best_src = np.full((N,), -1, dtype=np.int32)
    dst_best_src[best_dst] = best_src

    # apply MNN mask on original keep arrays
    mnn_mask = (dst_best_src[dst_keep] == src_keep)

    src_mnn = src_keep[mnn_mask]
    dst_mnn = dst_keep[mnn_mask]
    sim_mnn = sim_keep[mnn_mask]
    mar_mnn = mar_keep[mnn_mask]

    n_pairs_mnn = int(len(src_mnn))
    if n_pairs_mnn == 0:
        return {
            "grid_h": np.int32(gh), "grid_w": np.int32(gw),
            "has_peak": np.int8(0), "peak_ratio": np.float32(0.0),
            "peaks_dy_dx": np.zeros((0,2), dtype=np.int16),
            "peaks_weight": np.zeros((0,), dtype=np.float32),
            "peaks_count": np.zeros((0,), dtype=np.int32),
            "best_src": np.zeros((0,), dtype=np.int32),
            "best_dst": np.zeros((0,), dtype=np.int32),
            "best_sim": np.zeros((0,), dtype=np.float16),
            "best_weight": np.float32(0.0),
            "best_count": np.int32(0),
            "best_mean_sim": np.float32(0.0),
            "n_pairs_thr": np.int32(n_pairs_thr),
            "n_pairs_mnn": np.int32(0),
            "best_inlier_ratio": np.float32(0.0),
            "best_weight_frac": np.float32(0.0),
        }

    # offsets
    src_r = src_mnn // gw; src_c = src_mnn % gw
    dst_r = dst_mnn // gw; dst_c = dst_mnn % gw
    dy = (dst_r - src_r).astype(np.int32)
    dx = (dst_c - src_c).astype(np.int32)
    dyc, dxc = canonicalize_offsets(dy, dx)

    if bin_step > 1:
        dyb = (np.round(dyc / bin_step)).astype(np.int32) * bin_step
        dxb = (np.round(dxc / bin_step)).astype(np.int32) * bin_step
    else:
        dyb, dxb = dyc, dxc

    # vote weights
    w = np.clip(sim_mnn, 0.0, 1.0).astype(np.float32, copy=False)
    if wpow != 1.0:
        w = np.power(w, wpow, dtype=np.float32)

    m = np.clip(mar_mnn, 0.0, None).astype(np.float32, copy=False)
    if mpow != 1.0:
        m = np.power(m, mpow, dtype=np.float32)

    wv = (w * (m + 1e-9)).astype(np.float32, copy=False)

    keys = _encode_key(dyb, dxb)
    uniq, inv = np.unique(keys, return_inverse=True)
    sum_w = np.bincount(inv, weights=wv, minlength=len(uniq)).astype(np.float32)
    cnt   = np.bincount(inv, minlength=len(uniq)).astype(np.int32)

    order_pk = np.argsort(-sum_w)

    def _decode_key(key64: int):
        dy_pk = np.int32(key64 >> 32)
        dx_pk = np.int32(key64 & np.int64(0xffffffff))
        if dx_pk >= 2**31:
            dx_pk = dx_pk - 2**32
        return int(dy_pk), int(dx_pk)

    # Peak NMS
    peaks = []
    for idx in order_pk:
        if cnt[idx] < min_peak_count:
            continue
        dy_pk, dx_pk = _decode_key(int(uniq[idx]))
        ok = True
        for (pdy, pdx, _, _) in peaks:
            if max(abs(dy_pk - pdy), abs(dx_pk - pdx)) < peak_min_sep:
                ok = False
                break
        if not ok:
            continue
        peaks.append((dy_pk, dx_pk, float(sum_w[idx]), int(cnt[idx])))
        if len(peaks) >= peaks_M:
            break

    if len(peaks) == 0:
        return {
            "grid_h": np.int32(gh), "grid_w": np.int32(gw),
            "has_peak": np.int8(0), "peak_ratio": np.float32(0.0),
            "peaks_dy_dx": np.zeros((0,2), dtype=np.int16),
            "peaks_weight": np.zeros((0,), dtype=np.float32),
            "peaks_count": np.zeros((0,), dtype=np.int32),
            "best_src": np.zeros((0,), dtype=np.int32),
            "best_dst": np.zeros((0,), dtype=np.int32),
            "best_sim": np.zeros((0,), dtype=np.float16),
            "best_weight": np.float32(0.0),
            "best_count": np.int32(0),
            "best_mean_sim": np.float32(0.0),
            "n_pairs_thr": np.int32(n_pairs_thr),
            "n_pairs_mnn": np.int32(n_pairs_mnn),
            "best_inlier_ratio": np.float32(0.0),
            "best_weight_frac": np.float32(0.0),
        }

    w1 = peaks[0][2]
    w2 = peaks[1][2] if len(peaks) > 1 else 0.0
    peak_ratio = float(w1 / (w2 + 1e-9)) if w2 > 0 else float(1e9)

    best_dy, best_dx, best_weight, best_count = peaks[0]
    best_key = _encode_key(np.array([best_dy], dtype=np.int32), np.array([best_dx], dtype=np.int32))[0]
    same = (keys == best_key)

    best_src = src_mnn[same].astype(np.int32, copy=False)
    best_dst = dst_mnn[same].astype(np.int32, copy=False)
    best_sim = sim_mnn[same].astype(np.float16, copy=False)
    best_mean_sim = float(np.mean(sim_mnn[same])) if best_src.size > 0 else 0.0

    peaks_arr   = np.array([[p[0], p[1]] for p in peaks], dtype=np.int16)
    weights_arr = np.array([p[2] for p in peaks], dtype=np.float32)
    counts_arr  = np.array([p[3] for p in peaks], dtype=np.int32)

    total_w = float(np.sum(sum_w)) + 1e-9
    best_inlier_ratio = float(best_count / max(1, n_pairs_mnn))
    best_weight_frac  = float(best_weight / total_w)

    return {
        "grid_h": np.int32(gh),
        "grid_w": np.int32(gw),
        "has_peak": np.int8(1),
        "peak_ratio": np.float32(peak_ratio),
        "peaks_dy_dx": peaks_arr,
        "peaks_weight": weights_arr,
        "peaks_count": counts_arr,
        "best_src": best_src,
        "best_dst": best_dst,
        "best_sim": best_sim,
        "best_weight": np.float32(best_weight),
        "best_count": np.int32(best_count),
        "best_mean_sim": np.float32(best_mean_sim),
        "n_pairs_thr": np.int32(n_pairs_thr),
        "n_pairs_mnn": np.int32(n_pairs_mnn),
        "best_inlier_ratio": np.float32(best_inlier_ratio),
        "best_weight_frac": np.float32(best_weight_frac),
    }

# ----------------------------
# Build processing lists (lebih aman)
# ----------------------------
train_meta_cols = [c for c in ["sample_id","case_id","variant","fold","y_forged","image_path"] if c in df_train_all.columns]
train_meta = df_train_all[train_meta_cols].copy()
train_meta["sample_id"] = train_meta["sample_id"].astype(str)
train_meta = train_meta.drop_duplicates(subset=["sample_id"], keep="first").set_index("sample_id")

train_list = man_train.copy()
train_list["sample_id"] = train_list["sample_id"].astype(str)
# drop overlap cols sebelum join
overlap_cols = [c for c in train_meta.columns if c in train_list.columns]
if overlap_cols:
    train_list = train_list.drop(columns=overlap_cols)
train_list = train_list.join(train_meta, on="sample_id")

test_meta = df_test[["case_id","image_path"]].copy()
test_meta["case_id"] = test_meta["case_id"].astype(str)
test_meta = test_meta.drop_duplicates(subset=["case_id"], keep="first").set_index("case_id")

test_list = man_test.copy()
test_list["case_id"] = test_list["case_id"].astype(str)
overlap_cols = [c for c in test_meta.columns if c in test_list.columns]
if overlap_cols:
    test_list = test_list.drop(columns=overlap_cols)
test_list = test_list.join(test_meta, on="case_id")

sort_train_cols = [c for c in ["variant","fold","case_id","sample_id"] if c in train_list.columns]
train_list = train_list.sort_values(sort_train_cols).reset_index(drop=True)
test_list  = test_list.sort_values(["case_id"]).reset_index(drop=True)

if CFG_MATCH["limit_train"] is not None:
    train_list = train_list.iloc[:int(CFG_MATCH["limit_train"])].copy()
if CFG_MATCH["limit_test"] is not None:
    test_list = test_list.iloc[:int(CFG_MATCH["limit_test"])].copy()

print(f"\nTo process:")
print(f"  train samples: {len(train_list):,} (per sample_id)")
print(f"  test  cases  : {len(test_list):,} (per case_id)")

# ----------------------------
# Run matching (lebih cepat pakai itertuples)
# - Skip existing hanya jika cfg_id sama, jika beda -> rebuild
# ----------------------------
def run_block(df_list: pd.DataFrame, id_col: str, out_dir: Path, label: str):
    t0 = time.time()
    done = skipped = rebuilt = failed = 0

    # itertuples cepat
    cols = list(df_list.columns)
    for i, r in enumerate(df_list.itertuples(index=False), start=1):
        uid = str(getattr(r, id_col))
        feat_path = str(getattr(r, "feat_path"))
        outp = out_dir / f"{uid}.npz"

        if outp.exists() and CFG_MATCH["skip_if_exists"]:
            old = read_match_cfg_id(outp)
            if old == MATCH_CFG_ID:
                skipped += 1
                continue
            else:
                try:
                    outp.unlink()
                except Exception:
                    pass
                rebuilt += 1

        loaded = load_dino_patch_from_npz(feat_path)
        if loaded is None:
            failed += 1
            if failed <= 10:
                print(f"[{label}] WARN missing/invalid feat uid={uid} feat_path={feat_path}")
            continue

        feat_np, gh, gw = loaded
        try:
            payload = robust_match_one(feat_np, gh, gw, CFG_MATCH)

            # meta fields (tidak ganggu STAGE 4)
            payload["match_cfg_id"] = np.array(MATCH_CFG_ID, dtype=str)
            payload["uid"] = np.array(uid, dtype=str)
            if "case_id" in cols:
                payload["case_id"] = np.array(str(getattr(r, "case_id", "")), dtype=str)
            if "variant" in cols:
                payload["variant"] = np.array(str(getattr(r, "variant", "")), dtype=str)

            save_match_npz(outp, payload)
            done += 1

        except Exception as e:
            failed += 1
            if failed <= 10:
                print(f"[{label}] WARN fail uid={uid} err={type(e).__name__}: {e}")

        if (done > 0) and (done % int(CFG_MATCH["print_every"]) == 0):
            dt = time.time() - t0
            rate = done / max(dt, 1e-9)
            print(f"[{label}] done={done:,} skipped={skipped:,} rebuilt={rebuilt:,} failed={failed:,} | {rate:.2f} img/s | last={uid}")
            gc.collect()

    dt = time.time() - t0
    print(f"\n[{label}] SUMMARY")
    print(f"  done   : {done:,}")
    print(f"  skipped: {skipped:,}")
    print(f"  rebuilt: {rebuilt:,}")
    print(f"  failed : {failed:,}")
    print(f"  time_s : {dt:.1f}")
    return {"done": done, "skipped": skipped, "rebuilt": rebuilt, "failed": failed, "time_s": dt}

print("\n[1/2] Matching TRAIN_ALL ...")
rep_train = run_block(train_list, id_col="sample_id", out_dir=MATCH_TRAIN_DIR, label="train_all")

print("\n[2/2] Matching TEST ...")
rep_test = run_block(test_list, id_col="case_id", out_dir=MATCH_TEST_DIR, label="test")

# ----------------------------
# Write manifests for Stage 4
# ----------------------------
man_match_train = pd.DataFrame({
    "sample_id": train_list["sample_id"].astype(str).values,
    "case_id": train_list["case_id"].astype(str).fillna("").values if "case_id" in train_list.columns else np.array([""]*len(train_list)),
    "variant": train_list["variant"].astype(str).fillna("").values if "variant" in train_list.columns else np.array([""]*len(train_list)),
    "fold": train_list["fold"].fillna(-1).astype(int).values if "fold" in train_list.columns else np.full(len(train_list), -1, dtype=int),
    "y_forged": train_list["y_forged"].fillna(-1).astype(int).values if "y_forged" in train_list.columns else np.full(len(train_list), -1, dtype=int),
    "match_path": [str(MATCH_TRAIN_DIR / f"{sid}.npz") for sid in train_list["sample_id"].astype(str).values],
})
man_match_train["match_exists"] = man_match_train["match_path"].map(lambda p: Path(p).exists()).astype(int)

man_match_test = pd.DataFrame({
    "case_id": test_list["case_id"].astype(str).values,
    "match_path": [str(MATCH_TEST_DIR / f"{cid}.npz") for cid in test_list["case_id"].astype(str).values],
})
man_match_test["match_exists"] = man_match_test["match_path"].map(lambda p: Path(p).exists()).astype(int)

MATCH_MAN_TRAIN_PATH = MATCH_ROOT / "manifest_match_train_all.csv"
MATCH_MAN_TEST_PATH  = MATCH_ROOT / "manifest_match_test.csv"
man_match_train.to_csv(MATCH_MAN_TRAIN_PATH, index=False)
man_match_test.to_csv(MATCH_MAN_TEST_PATH, index=False)

with open(MATCH_ROOT / "match_summary.json", "w") as f:
    json.dump({"cfg": CFG_MATCH, "match_cfg_id": MATCH_CFG_ID, "train": rep_train, "test": rep_test}, f, indent=2)

print("\nWrote match manifests:")
print(" -", MATCH_MAN_TRAIN_PATH, "| exists_rate:", float(man_match_train["match_exists"].mean()) if len(man_match_train) else 0.0)
print(" -", MATCH_MAN_TEST_PATH,  "| exists_rate:", float(man_match_test["match_exists"].mean()) if len(man_match_test) else 0.0)
print(" -", MATCH_ROOT / "match_summary.json")

MATCH_CACHE_ROOT = str(MATCH_ROOT)
print("\nDONE. Exported: CFG_MATCH, MATCH_CFG_ID, MATCH_CACHE_ROOT, MATCH_TRAIN_DIR, MATCH_TEST_DIR, MATCH_MAN_TRAIN_PATH, MATCH_MAN_TEST_PATH")


# Verification, Mask Reconstruction & Postprocess (One Block)

In [ ]:
# ============================================================
# STAGE 4 — Verification, Mask Reconstruction & Postprocess (ONE CELL, REVISI FULL v6 - Precise + Recall-Safe)
# - Fix meta STAGE 2: supports resized_hw_base / resized_hw
# - Auto-find manifests STAGE 2/3 (support cfg_* + versioned match)
# - Skip only if cfg_hash match
# - Adaptive fallback: kalau grid kosong tapi match kuat -> relax threshold sedikit (tanpa overmask)
# - Output dir versioned by cfg_hash (anti ketimpa)
# ============================================================

import os, gc, time, json, hashlib
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

try:
    import scipy.ndimage as ndi
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ----------------------------
# REQUIRE
# ----------------------------
for need in ["df_train_all", "df_test"]:
    if need not in globals():
        raise RuntimeError(f"Missing: {need}. Jalankan STAGE 1 dulu (df_train_all & df_test).")

df_train_all = df_train_all.copy()
df_test = df_test.copy()

# ----------------------------
# Helper: normalize id -> string stabil (hindari '5506.0')
# ----------------------------
def _norm_one_id(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    if isinstance(x, (np.integer, int)):
        return str(int(x))
    if isinstance(x, (np.floating, float)):
        if np.isfinite(x) and abs(x - round(x)) < 1e-9:
            return str(int(round(x)))
        return str(float(x))
    s = str(x)
    if s.endswith(".0"):
        head = s[:-2]
        if head.isdigit():
            return head
    return s

def norm_id_series(s: pd.Series) -> pd.Series:
    return s.map(_norm_one_id)

for c in ["sample_id", "case_id"]:
    if c in df_train_all.columns:
        df_train_all[c] = norm_id_series(df_train_all[c])
if "case_id" in df_test.columns:
    df_test["case_id"] = norm_id_series(df_test["case_id"])

# ----------------------------
# Auto-find manifests (STAGE 2 / STAGE 3)
# ----------------------------
def _auto_find_latest(root: Path, filename: str) -> Path:
    root = Path(root)
    if not root.exists():
        return Path(filename)
    cands = list(root.rglob(filename))
    if not cands:
        return Path(filename)
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]

# STAGE 2 manifests
MAN_TRAIN_PATH = Path(str(globals().get("MAN_TRAIN_PATH", ""))).expanduser()
MAN_TEST_PATH  = Path(str(globals().get("MAN_TEST_PATH", ""))).expanduser()

if not str(MAN_TRAIN_PATH):
    MAN_TRAIN_PATH = Path("/kaggle/working/recodai_luc/cache/dino_v2_base/manifest_train_all.csv")
if not str(MAN_TEST_PATH):
    MAN_TEST_PATH = Path("/kaggle/working/recodai_luc/cache/dino_v2_base/manifest_test.csv")

if not MAN_TRAIN_PATH.exists():
    MAN_TRAIN_PATH = _auto_find_latest("/kaggle/working/recodai_luc/cache/dino_v2_base", "manifest_train_all.csv")
if not MAN_TEST_PATH.exists():
    MAN_TEST_PATH  = _auto_find_latest("/kaggle/working/recodai_luc/cache/dino_v2_base", "manifest_test.csv")

if not MAN_TRAIN_PATH.exists():
    raise FileNotFoundError(f"Manifest train STAGE 2 tidak ditemukan: {MAN_TRAIN_PATH}")
if not MAN_TEST_PATH.exists():
    raise FileNotFoundError(f"Manifest test STAGE 2 tidak ditemukan: {MAN_TEST_PATH}")

man_train = pd.read_csv(MAN_TRAIN_PATH)
man_test  = pd.read_csv(MAN_TEST_PATH)

need2_train = {"sample_id","feat_path","feat_exists"}
need2_test  = {"case_id","feat_path","feat_exists"}
if not need2_train.issubset(set(man_train.columns)):
    raise RuntimeError(f"manifest_train_all missing: {need2_train - set(man_train.columns)}")
if not need2_test.issubset(set(man_test.columns)):
    raise RuntimeError(f"manifest_test missing: {need2_test - set(man_test.columns)}")

man_train["sample_id"] = norm_id_series(man_train["sample_id"])
man_test["case_id"]    = norm_id_series(man_test["case_id"])
man_train = man_train[man_train["feat_exists"] == 1].copy()
man_test  = man_test[man_test["feat_exists"]  == 1].copy()

# STAGE 3 manifests
MATCH_MAN_TRAIN_PATH = globals().get("MATCH_MAN_TRAIN_PATH", None)
MATCH_MAN_TEST_PATH  = globals().get("MATCH_MAN_TEST_PATH", None)

if MATCH_MAN_TRAIN_PATH is not None and MATCH_MAN_TEST_PATH is not None:
    MATCH_MAN_TRAIN_PATH = Path(str(MATCH_MAN_TRAIN_PATH))
    MATCH_MAN_TEST_PATH  = Path(str(MATCH_MAN_TEST_PATH))
else:
    # cari yang terbaru di cache (support match_base_v3_cfg_* dll)
    MATCH_MAN_TRAIN_PATH = _auto_find_latest("/kaggle/working/recodai_luc/cache", "manifest_match_train_all.csv")
    MATCH_MAN_TEST_PATH  = _auto_find_latest("/kaggle/working/recodai_luc/cache", "manifest_match_test.csv")

if not MATCH_MAN_TRAIN_PATH.exists():
    raise FileNotFoundError(f"Manifest match train STAGE 3 tidak ditemukan: {MATCH_MAN_TRAIN_PATH}")
if not MATCH_MAN_TEST_PATH.exists():
    raise FileNotFoundError(f"Manifest match test STAGE 3 tidak ditemukan: {MATCH_MAN_TEST_PATH}")

man_match_train = pd.read_csv(MATCH_MAN_TRAIN_PATH)
man_match_test  = pd.read_csv(MATCH_MAN_TEST_PATH)

need3_train = {"sample_id","match_path","match_exists"}
need3_test  = {"case_id","match_path","match_exists"}
if not need3_train.issubset(set(man_match_train.columns)):
    raise RuntimeError(f"manifest_match_train_all missing: {need3_train - set(man_match_train.columns)}")
if not need3_test.issubset(set(man_match_test.columns)):
    raise RuntimeError(f"manifest_match_test missing: {need3_test - set(man_match_test.columns)}")

man_match_train["sample_id"] = norm_id_series(man_match_train["sample_id"])
man_match_test["case_id"]    = norm_id_series(man_match_test["case_id"])

# ----------------------------
# CONFIG (precision + recall-safe)
# ----------------------------
CFG_RECON = {
    # verification gates
    "sim_inlier_thr": 0.82,         # inlier sim threshold
    "min_pairs": 22,                # sedikit lebih longgar dari 25 (recall naik)
    "min_pairs_small": 14,          # untuk grid kecil / match kecil
    "small_grid_N": 18*18,          # kalau gh*gw <= ini -> pakai min_pairs_small

    # build grid from pairs
    "score_mix": 0.65,
    "grid_thr_q": 0.90,
    "min_count_q": 0.78,            # sedikit lebih longgar
    "thr_grid_floor": 0.38,         # floor sedikit turun agar tidak sering kosong
    "grid_dilate": 0,               # tetap tidak melebar (high precision)

    # adaptive relax (hanya jika grid kosong tapi match kuat)
    "enable_relax_if_empty": True,
    "relax_q_to": 0.82,             # turunkan quantile threshold jika kosong
    "relax_min_count_q_to": 0.60,

    # postprocess
    "do_open": True,
    "open_ks": 3,
    "do_close": True,
    "close_ks": 3,

    # component filter
    "min_area_frac": 0.0005,
    "keep_topk_components": 2,

    # overmask guard
    "max_area_frac": 0.32,

    # IO
    "skip_if_exists": True,         # skip hanya jika cfg_hash sama
    "print_every": 400,
}

_CFG_HASH = hashlib.md5(json.dumps(CFG_RECON, sort_keys=True).encode()).hexdigest()[:10]

# Output dirs (versioned by cfg_hash)
PRED_ROOT = Path("/kaggle/working/recodai_luc/cache") / f"pred_base_v2_v6_cfg_{_CFG_HASH}"
PRED_TRAIN_DIR = PRED_ROOT / "train_all"
PRED_TEST_DIR  = PRED_ROOT / "test"
PRED_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
PRED_TEST_DIR.mkdir(parents=True, exist_ok=True)

print("MAN_TRAIN_PATH:", MAN_TRAIN_PATH, "| rows(feat_exists=1):", len(man_train))
print("MAN_TEST_PATH :", MAN_TEST_PATH,  "| rows(feat_exists=1):", len(man_test))
print("MATCH_MAN_TRAIN_PATH:", MATCH_MAN_TRAIN_PATH, "| rows:", len(man_match_train))
print("MATCH_MAN_TEST_PATH :", MATCH_MAN_TEST_PATH,  "| rows:", len(man_match_test))
print("PRED_ROOT:", PRED_ROOT)
print("SCIPY available:", _HAS_SCIPY)

# ----------------------------
# Helpers
# ----------------------------
def _meta_to_str(x):
    if isinstance(x, np.ndarray):
        try:
            x = x.item()
        except Exception:
            x = x.reshape(-1)[0]
    if isinstance(x, (bytes, np.bytes_)):
        return x.decode("utf-8", errors="ignore")
    return str(x)

def _read_meta_from_feat_npz(feat_path: str):
    p = Path(feat_path)
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)
    if "meta" not in z.files:
        return None
    try:
        meta = json.loads(_meta_to_str(z["meta"]))
    except Exception:
        return None

    # STAGE 2 v3: resized_hw_base
    oh, ow = meta.get("orig_hw", [None, None])

    # support multiple key names
    rh_rw = meta.get("resized_hw_base", None)
    if rh_rw is None:
        rh_rw = meta.get("resized_hw", None)  # fallback legacy
    if rh_rw is None:
        # last: try proc dims from resized_hw_base keys
        rh, rw = None, None
    else:
        rh, rw = rh_rw[0], rh_rw[1]

    gh_gw = meta.get("grid_hw", [None, None])
    gh, gw = gh_gw[0], gh_gw[1]
    patch = meta.get("patch_size", None)

    if None in [oh, ow, rh, rw, gh, gw, patch]:
        return None
    return {
        "orig_h": int(oh), "orig_w": int(ow),
        "proc_h": int(rh), "proc_w": int(rw),
        "grid_h": int(gh), "grid_w": int(gw),
        "patch": int(patch),
    }

def _load_match_npz(match_path: str):
    p = Path(match_path)
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)

    out = {}
    keys = ["grid_h","grid_w","has_peak","peak_ratio","best_weight","best_count","best_mean_sim",
            "n_pairs_thr","n_pairs_mnn","best_src","best_dst","best_sim",
            "best_inlier_ratio","best_weight_frac"]
    for k in keys:
        if k in z.files:
            out[k] = z[k]

    def _s(v, default=0):
        if v is None:
            return default
        if isinstance(v, np.ndarray):
            if np.ndim(v) == 0:
                return v.item()
            return v.reshape(-1)[0].item()
        return v

    out["grid_h"] = int(_s(out.get("grid_h", None), 0))
    out["grid_w"] = int(_s(out.get("grid_w", None), 0))
    out["has_peak"] = int(_s(out.get("has_peak", None), 0))
    out["peak_ratio"] = float(_s(out.get("peak_ratio", None), 0.0))
    out["best_weight"] = float(_s(out.get("best_weight", None), 0.0))
    out["best_count"] = int(_s(out.get("best_count", None), 0))
    out["best_mean_sim"] = float(_s(out.get("best_mean_sim", None), 0.0))
    out["n_pairs_thr"] = int(_s(out.get("n_pairs_thr", None), 0))
    out["n_pairs_mnn"] = int(_s(out.get("n_pairs_mnn", None), 0))

    out["best_inlier_ratio"] = float(_s(out.get("best_inlier_ratio", None), 0.0))
    out["best_weight_frac"]  = float(_s(out.get("best_weight_frac", None), 0.0))

    out["best_src"] = out.get("best_src", np.zeros((0,), dtype=np.int32)).astype(np.int32, copy=False)
    out["best_dst"] = out.get("best_dst", np.zeros((0,), dtype=np.int32)).astype(np.int32, copy=False)
    out["best_sim"] = out.get("best_sim", np.zeros((0,), dtype=np.float16)).astype(np.float16, copy=False)
    return out

def pack_mask(mask_u8: np.ndarray) -> np.ndarray:
    m = (mask_u8 > 0).astype(np.uint8, copy=False).reshape(-1)
    return np.packbits(m, axis=None)

def _binary_close(mask: np.ndarray, ks: int) -> np.ndarray:
    if (not _HAS_SCIPY) or ks <= 1:
        return mask.astype(np.uint8)
    st = np.ones((ks, ks), dtype=bool)
    return ndi.binary_closing(mask.astype(bool), structure=st).astype(np.uint8)

def _binary_open(mask: np.ndarray, ks: int) -> np.ndarray:
    if (not _HAS_SCIPY) or ks <= 1:
        return mask.astype(np.uint8)
    st = np.ones((ks, ks), dtype=bool)
    return ndi.binary_opening(mask.astype(bool), structure=st).astype(np.uint8)

def _filter_components(mask: np.ndarray, min_area: int, keep_topk: int):
    mask = mask.astype(np.uint8, copy=False)
    if mask.sum() == 0:
        return mask, 0, 0

    if not _HAS_SCIPY:
        area = int(mask.sum())
        if area < int(min_area):
            return np.zeros_like(mask, dtype=np.uint8), 0, 0
        return mask, 1, area

    lab, n = ndi.label(mask.astype(bool))
    if n == 0:
        return np.zeros_like(mask, dtype=np.uint8), 0, 0

    areas = np.bincount(lab.ravel())
    areas[0] = 0

    comps = np.where(areas >= int(min_area))[0]
    if comps.size == 0:
        return np.zeros_like(mask, dtype=np.uint8), 0, 0

    comps = comps[np.argsort(areas[comps])[::-1]]
    if keep_topk is not None and int(keep_topk) > 0:
        comps = comps[:int(keep_topk)]

    out = np.isin(lab, comps).astype(np.uint8)
    largest = int(areas[comps[0]]) if comps.size else 0
    return out, int(comps.size), largest

def build_grid_from_pairs(best_src, best_dst, best_sim, gh, gw, cfg, force_relax=False):
    """
    Grid builder:
    - patch_score: sum(sim) on endpoints
    - patch_count: freq endpoints
    - comb = mix*score_norm + (1-mix)*count_norm
    - thr_used = max(floor, quantile(comb_nonzero, q))
    - cnt_thr_used = quantile(count_nonzero, min_count_q)
    - if force_relax: use relax_q_to & relax_min_count_q_to
    """
    N = int(gh * gw)
    if best_src is None or best_src.size == 0 or N <= 0:
        grid_score = np.zeros((max(gh,1), max(gw,1)), dtype=np.float32)
        grid_count = np.zeros((max(gh,1), max(gw,1)), dtype=np.uint16)
        mask_grid  = np.zeros((max(gh,1), max(gw,1)), dtype=np.uint8)
        feats = {"inlier_ratio": 0.0, "pair_count": 0, "uniq_src": 0, "uniq_dst": 0, "mean_sim": 0.0,
                 "thr_used": 1.0, "cnt_thr_used": 999999, "score_mix": float(cfg["score_mix"]), "relaxed": int(force_relax)}
        return grid_score, grid_count, mask_grid, feats

    src = best_src.astype(np.int32, copy=False)
    dst = best_dst.astype(np.int32, copy=False)
    sim = best_sim.astype(np.float32, copy=False)

    patch_score = np.zeros((N,), dtype=np.float32)
    patch_count = np.zeros((N,), dtype=np.int32)

    w = np.clip(sim, 0.0, 1.0)
    np.add.at(patch_score, src, w); np.add.at(patch_score, dst, w)
    np.add.at(patch_count, src, 1); np.add.at(patch_count, dst, 1)

    smax = float(patch_score.max()) if patch_score.size else 0.0
    score_norm = patch_score / (smax + 1e-9) if smax > 0 else patch_score

    cmax = float(patch_count.max()) if patch_count.size else 0.0
    count_norm = patch_count.astype(np.float32) / (cmax + 1e-9) if cmax > 0 else patch_count.astype(np.float32)

    mix = float(cfg["score_mix"])
    comb = mix * score_norm + (1.0 - mix) * count_norm

    comb_nz = comb[comb > 0]
    if comb_nz.size > 0:
        q = float(cfg["relax_q_to"] if force_relax else cfg["grid_thr_q"])
        thr_dyn = float(np.quantile(comb_nz, q))
    else:
        thr_dyn = 1.0
    thr_used = max(float(cfg["thr_grid_floor"]), thr_dyn)

    cnt_nz = patch_count[patch_count > 0]
    if cnt_nz.size > 0:
        qcnt = float(cfg["relax_min_count_q_to"] if force_relax else cfg["min_count_q"])
        cnt_thr = int(np.quantile(cnt_nz.astype(np.float32), qcnt))
        cnt_thr = max(1, cnt_thr)
    else:
        cnt_thr = 999999

    mask_flat = (comb >= thr_used) & (patch_count >= cnt_thr)
    mask_grid = mask_flat.reshape(gh, gw).astype(np.uint8)

    grid_score = comb.reshape(gh, gw).astype(np.float32)
    grid_count = patch_count.reshape(gh, gw).astype(np.uint16)

    inlier_thr = float(cfg["sim_inlier_thr"])
    inlier_ratio = float((sim >= inlier_thr).mean()) if sim.size else 0.0

    feats = {
        "inlier_ratio": inlier_ratio,
        "pair_count": int(sim.size),
        "uniq_src": int(np.unique(src).size),
        "uniq_dst": int(np.unique(dst).size),
        "mean_sim": float(sim.mean()) if sim.size else 0.0,
        "thr_used": float(thr_used),
        "cnt_thr_used": int(cnt_thr),
        "score_mix": float(mix),
        "relaxed": int(force_relax),
    }
    return grid_score, grid_count, mask_grid, feats

def grid_to_orig(mask_grid, patch, proc_h, proc_w, orig_h, orig_w):
    patch = max(int(patch), 1)
    # repeat lebih ringan dari kron
    mask_proc = np.repeat(np.repeat(mask_grid.astype(np.uint8), patch, axis=0), patch, axis=1)
    mask_proc = mask_proc[:max(proc_h,1), :max(proc_w,1)]
    if (proc_h, proc_w) != (orig_h, orig_w):
        im = Image.fromarray((mask_proc * 255).astype(np.uint8))
        im = im.resize((max(orig_w,1), max(orig_h,1)), resample=Image.NEAREST)
        mask_orig = (np.array(im) > 0).astype(np.uint8)
    else:
        mask_orig = mask_proc.astype(np.uint8)
    return mask_orig

def _as_save_scalar(v):
    if isinstance(v, (bool, np.bool_)):
        return np.int8(int(v))
    if isinstance(v, (int, np.integer)):
        return np.int32(int(v))
    if isinstance(v, (float, np.floating)):
        return np.float32(float(v))
    try:
        return np.float32(float(v))
    except Exception:
        return np.float32(0.0)

def save_pred_npz(out_path: Path, mask_orig: np.ndarray, grid_score: np.ndarray, grid_count: np.ndarray, scalars: dict):
    mask_pack = pack_mask(mask_orig)
    payload = dict(
        mask_pack=mask_pack.astype(np.uint8),
        mask_h=np.int32(mask_orig.shape[0]),
        mask_w=np.int32(mask_orig.shape[1]),
        grid_score=grid_score.astype(np.float16, copy=False),
        grid_count=grid_count.astype(np.uint16, copy=False),
        cfg_hash=np.array(_CFG_HASH, dtype=str),
    )
    for k, v in scalars.items():
        payload[k] = _as_save_scalar(v)
    np.savez_compressed(str(out_path), **payload)

def pred_cfg_hash_matches(pred_path: Path) -> bool:
    if not pred_path.exists():
        return False
    try:
        z = np.load(pred_path, allow_pickle=False)
        if "cfg_hash" not in z.files:
            return False
        h = z["cfg_hash"]
        if isinstance(h, np.ndarray):
            try:
                h = h.item()
            except Exception:
                h = str(h.reshape(-1)[0])
        return str(h) == str(_CFG_HASH)
    except Exception:
        return False

def load_pred_scalars_from_npz(pred_path: Path):
    if not pred_path.exists():
        return None
    z = np.load(pred_path, allow_pickle=False)
    out = {}
    keys = ["has_peak","peak_ratio","best_weight","best_count","best_mean_sim",
            "inlier_ratio","pair_count","uniq_src","uniq_dst","mean_sim",
            "area_frac","n_comp","largest_comp","n_pairs_thr","n_pairs_mnn",
            "thr_used","cnt_thr_used","relaxed_used","min_pairs_used",
            "best_inlier_ratio","best_weight_frac"]
    for k in keys:
        if k in z.files:
            v = z[k]
            if np.ndim(v) == 0:
                out[k] = float(v)
            else:
                out[k] = float(np.array(v).reshape(-1)[0])
        else:
            out[k] = 0.0
    return out

# ----------------------------
# Build worklists
# ----------------------------
work_train = man_match_train.merge(man_train[["sample_id","feat_path"]], on="sample_id", how="left")
need_cols_train = ["case_id","variant","fold","y_forged","image_path"]
miss = [c for c in need_cols_train if c not in work_train.columns]
if miss:
    take = ["sample_id"] + [c for c in miss if c in df_train_all.columns]
    work_train = work_train.merge(df_train_all[take], on="sample_id", how="left")

work_train["sample_id"] = norm_id_series(work_train["sample_id"])
if "case_id" in work_train.columns:
    work_train["case_id"] = norm_id_series(work_train["case_id"])
work_train["feat_path"]  = work_train["feat_path"].fillna("").astype(str)
work_train["match_path"] = work_train["match_path"].fillna("").astype(str)

work_test = man_match_test.merge(man_test[["case_id","feat_path"]], on="case_id", how="left")
if "image_path" not in work_test.columns and "case_id" in df_test.columns and "image_path" in df_test.columns:
    tmp = df_test[["case_id","image_path"]].copy()
    tmp["case_id"] = norm_id_series(tmp["case_id"])
    work_test = work_test.merge(tmp, on="case_id", how="left")

work_test["case_id"] = norm_id_series(work_test["case_id"])
work_test["feat_path"]  = work_test["feat_path"].fillna("").astype(str)
work_test["match_path"] = work_test["match_path"].fillna("").astype(str)

sort_train_cols = [c for c in ["variant","fold","case_id","sample_id"] if c in work_train.columns]
work_train = work_train.sort_values(sort_train_cols).reset_index(drop=True) if sort_train_cols else work_train.sort_values(["sample_id"]).reset_index(drop=True)
work_test  = work_test.sort_values(["case_id"]).reset_index(drop=True)

print("\nWork sizes:")
print("  train_all:", len(work_train), "| match_exists rate:", float(work_train["match_exists"].mean()) if len(work_train) else 0.0)
print("  test     :", len(work_test),  "| match_exists rate:", float(work_test["match_exists"].mean()) if len(work_test) else 0.0)

# ----------------------------
# Main loop
# ----------------------------
def run_pred_block(dfw: pd.DataFrame, id_col: str, out_dir: Path, label: str, collect_feat_rows: bool):
    t0 = time.time()
    done = skipped = failed = 0
    feat_rows = []

    cols = list(dfw.columns)

    for i, r in enumerate(dfw.itertuples(index=False), start=1):
        uid = str(getattr(r, id_col))
        outp = out_dir / f"{uid}.npz"

        # Skip only if file exists AND cfg_hash matches
        if CFG_RECON["skip_if_exists"] and outp.exists() and pred_cfg_hash_matches(outp):
            skipped += 1
            if collect_feat_rows:
                scal = load_pred_scalars_from_npz(outp) or {}
                feat_row = {"uid": uid, **scal}
                for c in ["case_id","variant","fold","y_forged"]:
                    if c in cols:
                        feat_row[c] = getattr(r, c, None)
                feat_rows.append(feat_row)
            continue

        feat_path  = str(getattr(r, "feat_path", ""))
        match_path = str(getattr(r, "match_path", ""))

        meta = _read_meta_from_feat_npz(feat_path) if feat_path else None
        mch  = _load_match_npz(match_path) if match_path else None

        # fallback meta dari image
        if meta is None:
            ip = str(getattr(r, "image_path", ""))
            if ip and Path(ip).exists():
                with Image.open(ip) as im:
                    im = im.convert("RGB")
                    ow, oh = im.size
                meta = {"orig_h": int(oh), "orig_w": int(ow),
                        "proc_h": int(oh), "proc_w": int(ow),
                        "grid_h": 1, "grid_w": 1, "patch": 1}
            else:
                meta = {"orig_h": 1, "orig_w": 1, "proc_h": 1, "proc_w": 1, "grid_h": 1, "grid_w": 1, "patch": 1}

        orig_h, orig_w = int(meta["orig_h"]), int(meta["orig_w"])
        gh, gw         = int(meta["grid_h"]), int(meta["grid_w"])
        proc_h, proc_w = int(meta["proc_h"]), int(meta["proc_w"])
        patch          = int(meta["patch"])

        mask_orig  = np.zeros((max(orig_h,1), max(orig_w,1)), dtype=np.uint8)
        grid_score = np.zeros((max(gh,1), max(gw,1)), dtype=np.float32)
        grid_count = np.zeros((max(gh,1), max(gw,1)), dtype=np.uint16)

        scalars = {
            "has_peak": 0,
            "peak_ratio": 0.0,
            "best_weight": 0.0,
            "best_count": 0,
            "best_mean_sim": 0.0,
            "n_pairs_thr": 0,
            "n_pairs_mnn": 0,
            "best_inlier_ratio": 0.0,
            "best_weight_frac": 0.0,

            "inlier_ratio": 0.0,
            "pair_count": 0,
            "uniq_src": 0,
            "uniq_dst": 0,
            "mean_sim": 0.0,

            "thr_used": 0.0,
            "cnt_thr_used": 0.0,
            "relaxed_used": 0,
            "min_pairs_used": 0,

            "area_frac": 0.0,
            "n_comp": 0,
            "largest_comp": 0,
        }

        try:
            if mch is not None:
                # align grid size jika match menyimpan grid_h/w
                gh_m = int(mch.get("grid_h", gh))
                gw_m = int(mch.get("grid_w", gw))
                if (gh_m > 0 and gw_m > 0) and (gh_m != gh or gw_m != gw):
                    gh, gw = gh_m, gw_m
                    grid_score = np.zeros((gh, gw), dtype=np.float32)
                    grid_count = np.zeros((gh, gw), dtype=np.uint16)

                scalars["has_peak"] = int(mch.get("has_peak", 0))
                scalars["peak_ratio"] = float(mch.get("peak_ratio", 0.0))
                scalars["best_weight"] = float(mch.get("best_weight", 0.0))
                scalars["best_count"] = int(mch.get("best_count", 0))
                scalars["best_mean_sim"] = float(mch.get("best_mean_sim", 0.0))
                scalars["n_pairs_thr"] = int(mch.get("n_pairs_thr", 0))
                scalars["n_pairs_mnn"] = int(mch.get("n_pairs_mnn", 0))
                scalars["best_inlier_ratio"] = float(mch.get("best_inlier_ratio", 0.0))
                scalars["best_weight_frac"]  = float(mch.get("best_weight_frac", 0.0))

                # adaptive min_pairs
                gridN = int(gh * gw)
                min_pairs_used = int(CFG_RECON["min_pairs_small"] if gridN <= int(CFG_RECON["small_grid_N"]) else CFG_RECON["min_pairs"])
                scalars["min_pairs_used"] = int(min_pairs_used)

                if scalars["has_peak"] == 1 and scalars["best_count"] >= min_pairs_used:
                    best_src = mch.get("best_src", np.zeros((0,), dtype=np.int32))
                    best_dst = mch.get("best_dst", np.zeros((0,), dtype=np.int32))
                    best_sim = mch.get("best_sim", np.zeros((0,), dtype=np.float16))

                    # first pass (strict)
                    grid_score, grid_count, mask_grid, vfeats = build_grid_from_pairs(
                        best_src, best_dst, best_sim, gh, gw, CFG_RECON, force_relax=False
                    )

                    # if empty but match seems strong -> relax (recall-safe)
                    if (mask_grid.sum() == 0) and bool(CFG_RECON["enable_relax_if_empty"]):
                        # strong heuristics: best_mean_sim cukup tinggi atau peak_ratio tinggi
                        strong = (scalars["best_mean_sim"] >= float(CFG_RECON["sim_inlier_thr"]) - 0.01) or (scalars["peak_ratio"] >= 1.25)
                        if strong:
                            grid_score, grid_count, mask_grid, vfeats = build_grid_from_pairs(
                                best_src, best_dst, best_sim, gh, gw, CFG_RECON, force_relax=True
                            )
                            scalars["relaxed_used"] = 1

                    scalars["thr_used"]     = float(vfeats.get("thr_used", 0.0))
                    scalars["cnt_thr_used"] = float(vfeats.get("cnt_thr_used", 0.0))

                    # upsample to orig
                    mask_orig = grid_to_orig(mask_grid, patch, proc_h, proc_w, orig_h, orig_w)

                    # postprocess
                    if CFG_RECON["do_open"]:
                        mask_orig = _binary_open(mask_orig, int(CFG_RECON["open_ks"]))
                    if CFG_RECON["do_close"]:
                        mask_orig = _binary_close(mask_orig, int(CFG_RECON["close_ks"]))

                    # component filter
                    min_area = int(float(CFG_RECON["min_area_frac"]) * float(mask_orig.size))
                    min_area = max(1, min_area)
                    mask_orig, n_comp, largest = _filter_components(
                        mask_orig, min_area=min_area, keep_topk=int(CFG_RECON["keep_topk_components"])
                    )

                    area_frac = float(mask_orig.sum()) / float(mask_orig.size + 1e-9)

                    # overmask guard
                    if area_frac > float(CFG_RECON["max_area_frac"]):
                        mask_orig[:] = 0
                        area_frac = 0.0
                        n_comp = 0
                        largest = 0

                    # verification stats
                    # inlier_ratio dihitung dari best_sim (lebih relevan daripada global sim_thr)
                    simf = best_sim.astype(np.float32, copy=False)
                    inlier_ratio = float((simf >= float(CFG_RECON["sim_inlier_thr"])).mean()) if simf.size else 0.0

                    scalars["inlier_ratio"] = float(inlier_ratio)
                    scalars["pair_count"]   = int(vfeats.get("pair_count", 0))
                    scalars["uniq_src"]     = int(vfeats.get("uniq_src", 0))
                    scalars["uniq_dst"]     = int(vfeats.get("uniq_dst", 0))
                    scalars["mean_sim"]     = float(vfeats.get("mean_sim", 0.0))
                    scalars["area_frac"]    = float(area_frac)
                    scalars["n_comp"]       = int(n_comp)
                    scalars["largest_comp"] = int(largest)

            save_pred_npz(outp, mask_orig, grid_score, grid_count, scalars)

            if collect_feat_rows:
                feat_row = {"uid": uid, **scalars}
                for c in ["case_id","variant","fold","y_forged"]:
                    if c in cols:
                        feat_row[c] = getattr(r, c, None)
                feat_rows.append(feat_row)

            done += 1
            if (done > 0) and (done % int(CFG_RECON["print_every"]) == 0):
                dt = time.time() - t0
                print(f"[{label}] done={done:,} skipped={skipped:,} failed={failed:,} | {done/max(dt,1e-9):.2f} item/s | last={uid}")
                gc.collect()

        except Exception as e:
            failed += 1
            if failed <= 10:
                print(f"[{label}] WARN fail uid={uid} err={type(e).__name__}: {e}")

    dt = time.time() - t0
    print(f"\n[{label}] SUMMARY")
    print(f"  done   : {done:,}")
    print(f"  skipped: {skipped:,}")
    print(f"  failed : {failed:,}")
    print(f"  time_s : {dt:.1f}")
    return feat_rows, {"done": done, "skipped": skipped, "failed": failed, "time_s": dt}

print("\n[1/2] Build preds for TRAIN_ALL (per sample_id) ...")
feat_rows_train, rep_train = run_pred_block(work_train, "sample_id", PRED_TRAIN_DIR, "train_all", collect_feat_rows=True)

print("\n[2/2] Build preds for TEST (per case_id) ...")
feat_rows_test, rep_test = run_pred_block(work_test, "case_id", PRED_TEST_DIR, "test", collect_feat_rows=False)

# ----------------------------
# Write pred manifests
# ----------------------------
man_pred_train = pd.DataFrame({
    "sample_id": work_train["sample_id"].astype(str).values,
    "case_id": work_train["case_id"].astype(str).fillna("").values if "case_id" in work_train.columns else [""]*len(work_train),
    "variant": work_train["variant"].astype(str).fillna("").values if "variant" in work_train.columns else [""]*len(work_train),
    "fold": work_train["fold"].fillna(-1).astype(int).values if "fold" in work_train.columns else [-1]*len(work_train),
    "y_forged": work_train["y_forged"].fillna(-1).astype(int).values if "y_forged" in work_train.columns else [-1]*len(work_train),
    "pred_path": [str(PRED_TRAIN_DIR / f"{sid}.npz") for sid in work_train["sample_id"].astype(str).values],
})
man_pred_train["pred_exists"] = man_pred_train["pred_path"].map(lambda p: Path(p).exists()).astype(int)

man_pred_test = pd.DataFrame({
    "case_id": work_test["case_id"].astype(str).values,
    "pred_path": [str(PRED_TEST_DIR / f"{cid}.npz") for cid in work_test["case_id"].astype(str).values],
})
man_pred_test["pred_exists"] = man_pred_test["pred_path"].map(lambda p: Path(p).exists()).astype(int)

PRED_MAN_TRAIN_PATH = PRED_ROOT / "manifest_pred_train_all.csv"
PRED_MAN_TEST_PATH  = PRED_ROOT / "manifest_pred_test.csv"
man_pred_train.to_csv(PRED_MAN_TRAIN_PATH, index=False)
man_pred_test.to_csv(PRED_MAN_TEST_PATH, index=False)

with open(PRED_ROOT / "pred_summary.json", "w") as f:
    json.dump({"cfg": CFG_RECON, "cfg_hash": _CFG_HASH, "train": rep_train, "test": rep_test}, f, indent=2)

df_pred_feat_train_all = pd.DataFrame(feat_rows_train) if len(feat_rows_train) else pd.DataFrame()

print("\nWrote pred manifests:")
print(" -", PRED_MAN_TRAIN_PATH, "| exists_rate:", float(man_pred_train["pred_exists"].mean()) if len(man_pred_train) else 0.0)
print(" -", PRED_MAN_TEST_PATH,  "| exists_rate:", float(man_pred_test["pred_exists"].mean()) if len(man_pred_test) else 0.0)
print(" -", PRED_ROOT / "pred_summary.json")

PRED_CACHE_ROOT = str(PRED_ROOT)
PRED_TRAIN_CACHE_DIR = str(PRED_TRAIN_DIR)
PRED_TEST_CACHE_DIR  = str(PRED_TEST_DIR)

print("\nDONE. Exported: CFG_RECON, PRED_CACHE_ROOT, PRED_TRAIN_CACHE_DIR, PRED_TEST_CACHE_DIR, PRED_MAN_TRAIN_PATH, PRED_MAN_TEST_PATH")
print("df_pred_feat_train_all head:")
print(df_pred_feat_train_all.head())


# Train & Save: Gate Model + Calibration + Thresholds

In [ ]:
# ============================================================
# STAGE 5 — DINOv2 Multi-Task (Classification + Segmentation) (ONE CELL, REVISI FULL v5)
# - Replace tabular gate with real DINOv2 training (multi-task)
# - Patch-grid segmentation (stable, fast) + classification head
# - CV OOF + calibration + threshold tuning (thr_forged, thr_mask, min_pred_patches)
# - Save ONLY trainable weights (heads + last N blocks) to keep checkpoint small
#
# Outputs:
# - /kaggle/working/recodai_luc/models/dinov2_mt_v5_*/checkpoints/fold_*.pt
# - model_config.json, thresholds.json, calibrator.joblib, oof_predictions.csv, report.json
# ============================================================

import os, json, time, math, ast, hashlib, random, gc
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import AutoModel
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression

# ----------------------------
# REQUIRE
# ----------------------------
if "df_train_all" not in globals():
    raise RuntimeError("Missing df_train_all. Jalankan STAGE 1 dulu.")
df_train_all = df_train_all.copy()

need_cols = {"sample_id","case_id","variant","fold","y_forged","mask_paths","image_path"}
miss = need_cols - set(df_train_all.columns)
if miss:
    raise RuntimeError(f"df_train_all missing columns: {miss}")

# ----------------------------
# ID normalization (same idea as STAGE 4)
# ----------------------------
def _norm_one_id(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    if isinstance(x, (np.integer, int)):
        return str(int(x))
    if isinstance(x, (np.floating, float)):
        if np.isfinite(x) and abs(x - round(x)) < 1e-9:
            return str(int(round(x)))
        return str(float(x))
    s = str(x)
    if s.endswith(".0"):
        head = s[:-2]
        if head.isdigit():
            return head
    return s

def norm_id_series(s: pd.Series) -> pd.Series:
    return s.map(_norm_one_id)

for c in ["sample_id","case_id"]:
    df_train_all[c] = norm_id_series(df_train_all[c])

df_train_all["variant"]  = df_train_all["variant"].astype(str)
df_train_all["fold"]     = df_train_all["fold"].astype(int)
df_train_all["y_forged"] = df_train_all["y_forged"].astype(int)
df_train_all["image_path"] = df_train_all["image_path"].astype(str)

# keep only fold >=0 (train CV)
df_train_all = df_train_all[df_train_all["fold"] >= 0].reset_index(drop=True)
if len(df_train_all) == 0:
    raise RuntimeError("Tidak ada data train dengan fold>=0.")

# ----------------------------
# CONFIG (tuning-ready)
# ----------------------------
CFG = {
    # DINO dir (from STAGE 1/2)
    "dino_dir": str(globals().get("DINO_BASE_DIR", "/kaggle/input/dinov2/pytorch/base/1")),

    # image / patch grid
    "img_size": 560,           # MUST be multiple of patch_size
    "patch_size": 14,
    "use_center_crop_val": True,

    # training
    "seed": 2025,
    "epochs": 5,
    "batch_size": 4,
    "num_workers": 2,
    "amp": True,
    "grad_accum": 1,
    "clip_grad": 1.0,

    # finetune policy
    "unfreeze_last_n_blocks": 2,   # 0 => linear probe (heads only)
    "use_grad_ckpt": False,

    # optimizer
    "lr_backbone": 3e-5,
    "lr_heads": 3e-4,
    "weight_decay": 0.05,

    # losses
    "w_cls": 1.0,
    "w_seg_bce": 1.0,
    "w_seg_dice": 1.0,
    "dice_eps": 1e-6,

    # augmentation (light, scientific-friendly)
    "aug_hflip_p": 0.5,
    "aug_vflip_p": 0.1,
    "aug_rot90_p": 0.15,         # rotate by 0/90/180/270
    "aug_brightness": 0.10,
    "aug_contrast": 0.10,

    # early stopping
    "patience": 2,

    # output
    "out_root": "/kaggle/working/recodai_luc/models",
    "gt_cache_root": "/kaggle/working/recodai_luc/cache/gt_grid_union",
    "print_every": 200,
}

IMG_SIZE = int(CFG["img_size"])
PATCH    = int(CFG["patch_size"])
assert IMG_SIZE % PATCH == 0, f"img_size harus kelipatan patch_size. Got {IMG_SIZE} vs {PATCH}"
GH = IMG_SIZE // PATCH
GW = IMG_SIZE // PATCH

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# threads (CPU side)
try:
    torch.set_num_threads(max(1, (os.cpu_count() or 2)//2))
except Exception:
    pass

# seed
def seed_everything(seed=2025):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(CFG["seed"])

# imagenet norm (DINOv2 typical)
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3,1,1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3,1,1)

# popcount LUT (for packed bits)
POPCNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

# ----------------------------
# Mask path parsing + loader (robust)
# ----------------------------
def parse_mask_paths(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    if isinstance(val, (list, tuple)):
        return [str(x) for x in val if str(x)]
    if isinstance(val, np.ndarray):
        try:
            return [str(x) for x in val.reshape(-1).tolist() if str(x)]
        except Exception:
            return []
    if isinstance(val, str):
        s = val.strip()
        if s == "" or s.lower() in ("nan","none","null"):
            return []
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                out = json.loads(s)
                if isinstance(out, (list, tuple)):
                    return [str(x) for x in out if str(x)]
            except Exception:
                pass
            try:
                out = ast.literal_eval(s)
                if isinstance(out, (list, tuple)):
                    return [str(x) for x in out if str(x)]
            except Exception:
                pass
        return [s]
    return []

def load_mask_any_as_bool(path: str, target_h: int, target_w: int):
    p = Path(str(path))
    if not p.exists():
        return None
    suf = p.suffix.lower()

    if suf in (".png",".jpg",".jpeg",".bmp",".tif",".tiff",".webp"):
        im = Image.open(p).convert("L")
        if im.size != (target_w, target_h):
            im = im.resize((target_w, target_h), resample=Image.NEAREST)
        return (np.array(im) > 0)

    if suf == ".npz":
        z = np.load(p, allow_pickle=False)
        if ("mask_pack" in z.files) and ("mask_h" in z.files) and ("mask_w" in z.files):
            mh = int(z["mask_h"]); mw = int(z["mask_w"])
            pack = z["mask_pack"].astype(np.uint8).reshape(-1)
            bits = np.unpackbits(pack, axis=None)[: mh*mw].reshape(mh, mw).astype(bool)
            if (mh, mw) != (target_h, target_w):
                im = Image.fromarray((bits.astype(np.uint8)*255)).resize((target_w, target_h), resample=Image.NEAREST)
                return (np.array(im) > 0)
            return bits
        return None

    # npy / others
    try:
        arr = np.load(p, allow_pickle=False)
    except Exception:
        arr = np.load(p, allow_pickle=True)
        if np.ndim(arr) == 0 and hasattr(arr, "item"):
            arr = arr.item()

    a = np.asarray(arr)
    a = np.squeeze(a)
    if a.ndim == 2:
        m = (a > 0)
        if m.shape != (target_h, target_w):
            im = Image.fromarray((m.astype(np.uint8)*255)).resize((target_w, target_h), resample=Image.NEAREST)
            m = (np.array(im) > 0)
        return m.astype(bool)

    if a.ndim == 3:
        # take max across channels
        m = (a.max(axis=-1) > 0)
        if m.shape != (target_h, target_w):
            im = Image.fromarray((m.astype(np.uint8)*255)).resize((target_w, target_h), resample=Image.NEAREST)
            m = (np.array(im) > 0)
        return m.astype(bool)

    return None

# ----------------------------
# GT cache: store union mask on PATCH GRID (GH x GW) packed bits
# - super small files, super fast training
# ----------------------------
GT_CACHE_ROOT = Path(CFG["gt_cache_root"]) / f"sz{IMG_SIZE}_p{PATCH}"
GT_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

def mask_to_grid(mask_bool_hw: np.ndarray, gh: int, gw: int, patch: int) -> np.ndarray:
    # mask_bool_hw: (H,W) where H=W=IMG_SIZE
    H, W = mask_bool_hw.shape
    assert H == gh*patch and W == gw*patch
    x = mask_bool_hw.reshape(gh, patch, gw, patch)
    g = x.max(axis=(1,3))  # (gh,gw) bool
    return g

def pack_grid_bool(grid_bool: np.ndarray) -> np.ndarray:
    flat = grid_bool.astype(np.uint8).reshape(-1)
    return np.packbits(flat, axis=None).astype(np.uint8)

def unpack_grid_pack(pack_u8: np.ndarray, gh: int, gw: int) -> np.ndarray:
    bits = np.unpackbits(pack_u8.reshape(-1).astype(np.uint8), axis=None)[: gh*gw]
    return bits.reshape(gh, gw).astype(np.uint8)

def build_gt_cache_for_row(sample_id: str, y_forged: int, mask_paths_val):
    outp = GT_CACHE_ROOT / f"{sample_id}.npz"
    if outp.exists():
        return True

    # authentic => empty
    if int(y_forged) == 0:
        pack = pack_grid_bool(np.zeros((GH, GW), dtype=bool))
        np.savez_compressed(str(outp),
                            grid_pack=pack,
                            gh=np.int16(GH), gw=np.int16(GW),
                            pos=np.int32(0),
                            empty=np.int8(1))
        return True

    paths = parse_mask_paths(mask_paths_val)
    if len(paths) == 0:
        pack = pack_grid_bool(np.zeros((GH, GW), dtype=bool))
        np.savez_compressed(str(outp),
                            grid_pack=pack,
                            gh=np.int16(GH), gw=np.int16(GW),
                            pos=np.int32(0),
                            empty=np.int8(1))
        return True

    union = np.zeros((IMG_SIZE, IMG_SIZE), dtype=bool)
    for p in paths:
        m = load_mask_any_as_bool(p, IMG_SIZE, IMG_SIZE)
        if m is None:
            continue
        union |= m

    grid = mask_to_grid(union, GH, GW, PATCH)
    pos = int(grid.sum())
    pack = pack_grid_bool(grid)

    np.savez_compressed(str(outp),
                        grid_pack=pack,
                        gh=np.int16(GH), gw=np.int16(GW),
                        pos=np.int32(pos),
                        empty=np.int8(1 if pos == 0 else 0))
    return True

print("\n[GT CACHE] Building missing GT grid cache files (only if absent) ...")
t0 = time.time()
built = 0
fail = 0
for i, r in enumerate(df_train_all.itertuples(index=False), start=1):
    sid = getattr(r, "sample_id")
    yfg = getattr(r, "y_forged")
    mpv = getattr(r, "mask_paths")
    try:
        ok = build_gt_cache_for_row(str(sid), int(yfg), mpv)
        built += 1 if ok else 0
    except Exception:
        fail += 1
    if (i % CFG["print_every"]) == 0:
        print(f"[GT CACHE] {i:,}/{len(df_train_all):,} processed | fail={fail:,}")
dt = time.time() - t0
print(f"[GT CACHE] Done. processed={built:,} fail={fail:,} elapsed_s={dt:.1f} | dir={GT_CACHE_ROOT}")

# ----------------------------
# Dataset
# ----------------------------
def pil_load_rgb(path: str):
    p = Path(path)
    if not p.exists():
        return None
    try:
        return Image.open(p).convert("RGB")
    except Exception:
        return None

def pil_resize_sq(img: Image.Image, size: int):
    # keep it simple: resize to square (scientific images often ok)
    return img.resize((size, size), resample=Image.BICUBIC)

def img_to_tensor_norm(img: Image.Image):
    arr = np.array(img, dtype=np.float32) / 255.0  # HWC
    t = torch.from_numpy(arr).permute(2,0,1).contiguous()  # CHW
    t = (t - IMNET_MEAN) / IMNET_STD
    return t

def apply_aug(image_t: torch.Tensor, grid_u8: torch.Tensor, cfg: dict):
    # image_t: [3,H,W], grid_u8: [GH,GW]
    # flips/rot90 on both; color jitter only on image
    if random.random() < float(cfg["aug_hflip_p"]):
        image_t = torch.flip(image_t, dims=[2])
        grid_u8 = torch.flip(grid_u8, dims=[1])
    if random.random() < float(cfg["aug_vflip_p"]):
        image_t = torch.flip(image_t, dims=[1])
        grid_u8 = torch.flip(grid_u8, dims=[0])

    if random.random() < float(cfg["aug_rot90_p"]):
        k = random.randint(0, 3)
        if k > 0:
            image_t = torch.rot90(image_t, k=k, dims=[1,2])
            grid_u8 = torch.rot90(grid_u8, k=k, dims=[0,1])

    # light brightness/contrast
    b = float(cfg["aug_brightness"])
    c = float(cfg["aug_contrast"])
    if (b > 0) or (c > 0):
        # brightness: multiply
        if b > 0 and random.random() < 0.5:
            factor = 1.0 + random.uniform(-b, b)
            image_t = torch.clamp(image_t * factor, -5.0, 5.0)
        # contrast: scale deviation from mean
        if c > 0 and random.random() < 0.5:
            mean = image_t.mean(dim=(1,2), keepdim=True)
            factor = 1.0 + random.uniform(-c, c)
            image_t = torch.clamp((image_t - mean) * factor + mean, -5.0, 5.0)

    return image_t, grid_u8

class DinoMTDataset(Dataset):
    def __init__(self, df: pd.DataFrame, train: bool):
        self.df = df.reset_index(drop=True)
        self.train = bool(train)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        sid = str(r["sample_id"])
        yfg = int(r["y_forged"])
        ip  = str(r["image_path"])

        img = pil_load_rgb(ip)
        if img is None:
            # fallback blank
            img = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
        img = pil_resize_sq(img, IMG_SIZE)
        x = img_to_tensor_norm(img)  # [3,IMG,IMG]

        gt_path = GT_CACHE_ROOT / f"{sid}.npz"
        if gt_path.exists():
            z = np.load(gt_path, allow_pickle=False)
            pack = z["grid_pack"].astype(np.uint8).reshape(-1)
            grid = unpack_grid_pack(pack, GH, GW)  # uint8 0/1
        else:
            grid = np.zeros((GH, GW), dtype=np.uint8)

        g = torch.from_numpy(grid).to(torch.uint8)  # [GH,GW]

        if self.train:
            x, g = apply_aug(x, g, CFG)

        # label float
        y = torch.tensor([float(yfg)], dtype=torch.float32)

        # seg target float (0/1)
        g = g.to(torch.float32)

        return x, g, y, sid

# ----------------------------
# Build DINOv2 multitask model
# ----------------------------
def find_encoder_layers(model):
    # try common structures
    if hasattr(model, "encoder") and hasattr(model.encoder, "layer"):
        return model.encoder.layer
    if hasattr(model, "vit") and hasattr(model.vit, "encoder") and hasattr(model.vit.encoder, "layer"):
        return model.vit.encoder.layer
    if hasattr(model, "backbone") and hasattr(model.backbone, "encoder") and hasattr(model.backbone.encoder, "layer"):
        return model.backbone.encoder.layer
    return None

def infer_num_layers_from_names(model):
    # fallback by scanning parameter names like "...encoder.layer.11...."
    ids = set()
    for n, _ in model.named_parameters():
        if ".layer." in n:
            try:
                t = n.split(".layer.")[1]
                i = int(t.split(".")[0])
                ids.add(i)
            except Exception:
                pass
    return (max(ids)+1) if ids else 0

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

def unfreeze_last_n_blocks(model, n_last: int):
    if n_last <= 0:
        return 0

    layers = find_encoder_layers(model)
    if layers is not None:
        L = len(layers)
        n_last = min(int(n_last), L)
        for i in range(L - n_last, L):
            for p in layers[i].parameters():
                p.requires_grad = True
        return n_last

    # fallback: infer number of layers from names
    L = infer_num_layers_from_names(model)
    if L <= 0:
        return 0
    n_last = min(int(n_last), L)
    allow = set(range(L - n_last, L))
    for name, p in model.named_parameters():
        if ".layer." in name:
            try:
                i = int(name.split(".layer.")[1].split(".")[0])
                if i in allow:
                    p.requires_grad = True
            except Exception:
                pass
    return n_last

class DinoMultiTask(nn.Module):
    def __init__(self, backbone: nn.Module, gh: int, gw: int):
        super().__init__()
        self.backbone = backbone
        self.gh = int(gh)
        self.gw = int(gw)

        # infer embed dim
        with torch.no_grad():
            dummy = torch.zeros((1,3,IMG_SIZE,IMG_SIZE), dtype=torch.float32)
            out = self.backbone(pixel_values=dummy)
            D = int(out.last_hidden_state.shape[-1])

        self.embed_dim = D

        # heads
        self.cls_head = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, 1)
        )
        self.seg_head = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, 1)  # per patch
        )

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        h = out.last_hidden_state  # [B, 1+N, D]
        cls_tok = h[:, 0, :]       # [B,D]
        ptok    = h[:, 1:, :]      # [B,N,D]

        cls_logit = self.cls_head(cls_tok).squeeze(-1)  # [B]

        # patch logits -> grid
        seg_patch = self.seg_head(ptok).squeeze(-1)     # [B,N]
        B, N = seg_patch.shape
        # expect N == gh*gw, but be robust
        if N != self.gh * self.gw:
            # try to infer grid close to square
            side = int(round(math.sqrt(N)))
            gh = max(1, side)
            gw = max(1, N // gh)
            gh = max(1, N // gw)
            seg_grid = seg_patch[:, :gh*gw].reshape(B, gh, gw)
        else:
            seg_grid = seg_patch.reshape(B, self.gh, self.gw)

        return cls_logit, seg_grid

def bce_with_pos_weight(logits, targets, pos_weight: float):
    pw = torch.tensor([pos_weight], device=logits.device, dtype=logits.dtype)
    return F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pw)

def dice_loss_from_logits(seg_logits, seg_targets, eps=1e-6):
    # seg_logits: [B,GH,GW], seg_targets: [B,GH,GW] (0/1)
    p = torch.sigmoid(seg_logits)
    t = seg_targets
    # flatten
    p = p.reshape(p.size(0), -1)
    t = t.reshape(t.size(0), -1)
    inter = (p * t).sum(dim=1)
    den = p.sum(dim=1) + t.sum(dim=1)
    dice = (2.0 * inter + eps) / (den + eps)
    return (1.0 - dice).mean()

def get_trainable_state_dict(model: nn.Module):
    sd = model.state_dict()
    trainable = {}
    # save params that are trainable OR belong to heads
    trainable_prefix = ("cls_head.", "seg_head.")
    for n, p in model.named_parameters():
        if p.requires_grad or n.startswith(trainable_prefix):
            trainable[n] = sd[n].detach().half().cpu()
    # also include LN running buffers if any inside heads (usually none)
    for n, b in model.named_buffers():
        if n.startswith(trainable_prefix):
            trainable[n] = b.detach().half().cpu() if torch.is_floating_point(b) else b.detach().cpu()
    return trainable

# ----------------------------
# Prepare data splits + weights
# ----------------------------
folds_all = df_train_all["fold"].values.astype(int)
unique_folds = sorted(np.unique(folds_all).tolist())

y_all = df_train_all["y_forged"].values.astype(int)
pos = float((y_all == 1).sum()); neg = float((y_all == 0).sum())
pos_weight_cls = (neg / max(pos, 1.0)) if (pos > 0 and neg > 0) else 1.0

# seg pos_weight: estimate from cached GT grids
# (neg_pixels / pos_pixels) on grid space
print("\n[SEG POS WEIGHT] Estimating from cached GT grids ...")
pos_pix = 0
tot_pix = 0
for sid in df_train_all["sample_id"].tolist():
    p = GT_CACHE_ROOT / f"{sid}.npz"
    if not p.exists():
        continue
    z = np.load(p, allow_pickle=False)
    pack = z["grid_pack"].astype(np.uint8).reshape(-1)
    cnt = int(POPCNT[pack].sum())
    pos_pix += cnt
    tot_pix += (GH * GW)
neg_pix = max(0, tot_pix - pos_pix)
pos_weight_seg = (neg_pix / max(pos_pix, 1)) if pos_pix > 0 else 1.0
pos_weight_seg = float(np.clip(pos_weight_seg, 1.0, 50.0))  # clamp to avoid extreme
print(f"[SEG POS WEIGHT] pos_pix={pos_pix:,} tot_pix={tot_pix:,} => pos_weight_seg={pos_weight_seg:.3f}")

# sampler weights (balance class)
w_pos = (pos + neg) / (2.0 * max(pos, 1.0))
w_neg = (pos + neg) / (2.0 * max(neg, 1.0))
sample_w = np.where(y_all == 1, w_pos, w_neg).astype(np.float32)

# ----------------------------
# Training / Eval helpers
# ----------------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_sid = []
    all_y = []
    all_p = []
    all_seg = []
    all_gt = []

    for x, g, y, sid in loader:
        x = x.to(device, non_blocking=True)
        g = g.to(device, non_blocking=True)  # [B,GH,GW]
        y = y.to(device, non_blocking=True).view(-1)  # [B]

        cls_logit, seg_logit = model(x)  # seg_logit [B,GH,GW] (or inferred)
        p = torch.sigmoid(cls_logit).detach().cpu().numpy().astype(np.float32)

        seg_p = torch.sigmoid(seg_logit).detach().cpu().numpy().astype(np.float16)

        all_sid.extend(list(sid))
        all_y.append(y.detach().cpu().numpy().astype(np.int8))
        all_p.append(p)
        all_seg.append(seg_p)
        all_gt.append(g.detach().cpu().numpy().astype(np.uint8))

    all_y = np.concatenate(all_y, axis=0)
    all_p = np.concatenate(all_p, axis=0)
    all_seg = np.concatenate(all_seg, axis=0)  # [N,gh,gw]
    all_gt = np.concatenate(all_gt, axis=0)    # [N,gh,gw] uint8

    # metrics
    if np.unique(all_y).size >= 2:
        auc = float(roc_auc_score(all_y, all_p))
        ap  = float(average_precision_score(all_y, all_p))
    else:
        auc = float("nan")
        ap  = float("nan")

    # dice at default thr=0.5 (for monitoring)
    thr = 0.5
    pred = (all_seg >= thr).astype(np.uint8)
    gt = (all_gt > 0).astype(np.uint8)

    ps = pred.reshape(len(pred), -1).sum(axis=1).astype(np.float32)
    gs = gt.reshape(len(gt), -1).sum(axis=1).astype(np.float32)
    inter = (pred & gt).reshape(len(pred), -1).sum(axis=1).astype(np.float32)

    dice = np.zeros(len(pred), dtype=np.float32)
    both0 = (ps == 0) & (gs == 0)
    dice[both0] = 1.0
    m = ~both0
    dice[m] = (2.0*inter[m]) / (ps[m] + gs[m] + 1e-6)

    # focus dice on forged (more meaningful)
    forged = (all_y == 1)
    dice_forg = float(dice[forged].mean()) if forged.any() else 0.0

    # combined score for checkpointing
    # (favor classification, but keep seg quality)
    score = float((0.65 * (auc if np.isfinite(auc) else 0.0)) + (0.35 * dice_forg))

    return {
        "sid": all_sid,
        "y": all_y,
        "p_cls": all_p,
        "seg_prob": all_seg,   # float16
        "gt_grid": all_gt,     # uint8
        "auc": auc,
        "ap": ap,
        "dice_forg@0.5": dice_forg,
        "score": score,
    }

def build_optimizer(model: nn.Module):
    # param groups: heads (higher lr), trainable backbone (lower lr)
    head_params = []
    bb_params = []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("cls_head.") or n.startswith("seg_head."):
            head_params.append(p)
        else:
            bb_params.append(p)

    groups = []
    if bb_params:
        groups.append({"params": bb_params, "lr": float(CFG["lr_backbone"]), "weight_decay": float(CFG["weight_decay"])})
    if head_params:
        groups.append({"params": head_params, "lr": float(CFG["lr_heads"]), "weight_decay": float(CFG["weight_decay"])})

    opt = torch.optim.AdamW(groups)
    return opt

def cosine_lr(step, total, lr_max):
    # simple cosine from lr_max -> 0
    if total <= 1:
        return lr_max
    t = min(max(step / total, 0.0), 1.0)
    return lr_max * 0.5 * (1.0 + math.cos(math.pi * t))

# ----------------------------
# Output directory (versioned)
# ----------------------------
RUN_TAG = hashlib.md5(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:10]
OUT_DIR = Path(CFG["out_root"]) / f"dinov2_mt_v5_{RUN_TAG}"
CKPT_DIR = OUT_DIR / "checkpoints"
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print("\nOUT_DIR:", OUT_DIR)
print("device:", device, "| folds:", unique_folds)

# ----------------------------
# CV TRAIN
# ----------------------------
n_all = len(df_train_all)
oof_p_cls = np.zeros(n_all, dtype=np.float32)
oof_seg   = np.zeros((n_all, GH, GW), dtype=np.float16)
oof_gt    = np.zeros((n_all, GH, GW), dtype=np.uint8)
oof_y     = df_train_all["y_forged"].values.astype(np.int8)
oof_fold  = df_train_all["fold"].values.astype(int)
oof_sid   = df_train_all["sample_id"].astype(str).tolist()

fold_reports = {}
best_fold_paths = {}

for f in unique_folds:
    print(f"\n====================\nFOLD {f}\n====================")
    tr_idx = np.where(oof_fold != f)[0]
    va_idx = np.where(oof_fold == f)[0]
    if len(va_idx) == 0:
        print("Skip fold (no val).")
        continue

    df_tr = df_train_all.iloc[tr_idx].reset_index(drop=True)
    df_va = df_train_all.iloc[va_idx].reset_index(drop=True)

    ds_tr = DinoMTDataset(df_tr, train=True)
    ds_va = DinoMTDataset(df_va, train=False)

    # sampler for train
    sw = sample_w[tr_idx]
    sampler = WeightedRandomSampler(weights=torch.from_numpy(sw), num_samples=len(sw), replacement=True)

    dl_tr = DataLoader(ds_tr, batch_size=int(CFG["batch_size"]), sampler=sampler,
                       num_workers=int(CFG["num_workers"]), pin_memory=(device.type=="cuda"))
    dl_va = DataLoader(ds_va, batch_size=int(CFG["batch_size"]), shuffle=False,
                       num_workers=int(CFG["num_workers"]), pin_memory=(device.type=="cuda"))

    # backbone
    print("Loading DINO from:", CFG["dino_dir"])
    backbone = AutoModel.from_pretrained(str(CFG["dino_dir"]), local_files_only=True)
    backbone.eval()

    if CFG.get("use_grad_ckpt", False) and hasattr(backbone, "gradient_checkpointing_enable"):
        try:
            backbone.gradient_checkpointing_enable()
            print("gradient checkpointing: ON")
        except Exception:
            pass

    freeze_all(backbone)
    used_unfreeze = unfreeze_last_n_blocks(backbone, int(CFG["unfreeze_last_n_blocks"]))
    print("unfreeze_last_n_blocks used:", used_unfreeze)

    model = DinoMultiTask(backbone, gh=GH, gw=GW).to(device)
    # heads always trainable
    for p in model.cls_head.parameters():
        p.requires_grad = True
    for p in model.seg_head.parameters():
        p.requires_grad = True

    opt = build_optimizer(model)

    # AMP scaler
    use_amp = bool(CFG["amp"]) and (device.type == "cuda")
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # training steps for cosine schedule
    steps_per_epoch = max(1, len(dl_tr))
    total_steps = int(CFG["epochs"]) * steps_per_epoch

    best_score = -1e9
    best_state = None
    best_epoch = -1
    bad_epochs = 0

    global_step = 0
    t0 = time.time()

    for epoch in range(int(CFG["epochs"])):
        model.train()
        running = 0.0
        n_seen = 0

        for it, (x, g, y, sid) in enumerate(dl_tr, start=1):
            x = x.to(device, non_blocking=True)
            g = g.to(device, non_blocking=True)         # [B,GH,GW]
            y = y.to(device, non_blocking=True).view(-1)  # [B]

            # lr schedule (two groups): scale by cosine on each group's base lr
            lr_scale = cosine_lr(global_step, total_steps, 1.0)
            for pg in opt.param_groups:
                base = float(pg.get("lr", 1e-4))
                # base already set; rescale relative to initial? keep simple:
                pg["lr"] = base * lr_scale

            with torch.cuda.amp.autocast(enabled=use_amp):
                cls_logit, seg_logit = model(x)

                # cls loss
                loss_cls = bce_with_pos_weight(cls_logit, y, pos_weight=pos_weight_cls)

                # seg loss on grid
                loss_seg_bce  = bce_with_pos_weight(seg_logit, g, pos_weight=pos_weight_seg)
                loss_seg_dice = dice_loss_from_logits(seg_logit, g, eps=float(CFG["dice_eps"]))

                loss = (float(CFG["w_cls"]) * loss_cls
                        + float(CFG["w_seg_bce"]) * loss_seg_bce
                        + float(CFG["w_seg_dice"]) * loss_seg_dice)

            loss = loss / float(CFG["grad_accum"])

            scaler.scale(loss).backward()

            if (it % int(CFG["grad_accum"])) == 0:
                # grad clip
                if float(CFG["clip_grad"]) > 0:
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), float(CFG["clip_grad"]))

                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)

            running += float(loss.item()) * float(CFG["grad_accum"])
            n_seen += x.size(0)
            global_step += 1

            if (it % int(CFG["print_every"])) == 0:
                dt = time.time() - t0
                print(f"epoch={epoch+1}/{CFG['epochs']} it={it}/{len(dl_tr)} "
                      f"loss={running/max(1,it):.4f} seen={n_seen} step={global_step} elapsed_s={dt:.1f}")

        # eval
        ev = evaluate(model, dl_va)
        print(f"[VAL] epoch={epoch+1} score={ev['score']:.5f} AUC={ev['auc']:.5f} AP={ev['ap']:.5f} dice_forg@0.5={ev['dice_forg@0.5']:.5f}")

        if ev["score"] > best_score:
            best_score = ev["score"]
            best_epoch = epoch + 1
            bad_epochs = 0
            # save trainable state
            best_state = get_trainable_state_dict(model)
        else:
            bad_epochs += 1
            if bad_epochs >= int(CFG["patience"]):
                print(f"Early stop on fold {f} at epoch {epoch+1}. Best epoch={best_epoch} best_score={best_score:.5f}")
                break

        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    # load best state into model (for OOF extraction)
    if best_state is not None:
        # apply into current model
        cur = model.state_dict()
        for k, v in best_state.items():
            if k in cur:
                cur[k] = v.to(cur[k].dtype)
        model.load_state_dict(cur, strict=False)

    # final val for OOF storing
    ev = evaluate(model, dl_va)

    # write OOF to global arrays
    # map val local indices -> global indices
    for j, sid in enumerate(ev["sid"]):
        # sid belongs to df_va order, map by position j
        gi = va_idx[j]
        oof_p_cls[gi] = ev["p_cls"][j]
        # ensure shapes align
        sg = ev["seg_prob"][j]
        gt = ev["gt_grid"][j]
        if sg.shape != (GH, GW):
            # resize fallback via nearest on numpy (rare)
            sg = np.array(Image.fromarray(sg.astype(np.float32)).resize((GW, GH), resample=Image.NEAREST)).astype(np.float16)
        if gt.shape != (GH, GW):
            gt = np.array(Image.fromarray(gt.astype(np.uint8)*255).resize((GW, GH), resample=Image.NEAREST) > 0).astype(np.uint8)
        oof_seg[gi] = sg
        oof_gt[gi]  = gt

    # save fold checkpoint (trainable only)
    ckpt_path = CKPT_DIR / f"fold_{int(f)}.pt"
    torch.save({
        "fold": int(f),
        "best_epoch": int(best_epoch),
        "best_score": float(best_score),
        "trainable_state": best_state,
        "unfreeze_last_n_blocks": int(CFG["unfreeze_last_n_blocks"]),
        "img_size": int(IMG_SIZE),
        "patch_size": int(PATCH),
        "gh": int(GH),
        "gw": int(GW),
        "dino_dir": str(CFG["dino_dir"]),
    }, ckpt_path)

    best_fold_paths[int(f)] = str(ckpt_path)

    fold_reports[int(f)] = {
        "best_epoch": int(best_epoch),
        "best_score": float(best_score),
        "val_auc": float(ev["auc"]) if np.isfinite(ev["auc"]) else None,
        "val_ap": float(ev["ap"]) if np.isfinite(ev["ap"]) else None,
        "val_dice_forg@0.5": float(ev["dice_forg@0.5"]),
        "ckpt_path": str(ckpt_path),
    }

    print(f"[FOLD {f}] saved: {ckpt_path}")

# ----------------------------
# Calibration on OOF (cls only)
# ----------------------------
y = oof_y.astype(int)
p_raw = oof_p_cls.astype(np.float32)

calibrator = None
calib_kind = "none"
try:
    if np.unique(y).size >= 2 and len(y) >= 200 and np.unique(p_raw).size >= 50:
        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(p_raw, y)
        calibrator = iso
        calib_kind = "isotonic"
    else:
        raise RuntimeError("Not enough unique probs for isotonic.")
except Exception:
    try:
        platt = LogisticRegression(solver="lbfgs", max_iter=4000)
        platt.fit(p_raw.reshape(-1,1), y)
        calibrator = platt
        calib_kind = "platt"
    except Exception:
        calibrator = None
        calib_kind = "none"

def apply_calibrator(p):
    p = np.asarray(p, dtype=np.float32)
    if calibrator is None or calib_kind == "none":
        return p
    if calib_kind == "isotonic":
        return calibrator.transform(p).astype(np.float32)
    return calibrator.predict_proba(p.reshape(-1,1))[:,1].astype(np.float32)

p_cal = apply_calibrator(p_raw)

# OOF cls metrics
if np.unique(y).size >= 2:
    auc_raw = float(roc_auc_score(y, p_raw))
    ap_raw  = float(average_precision_score(y, p_raw))
    auc_cal = float(roc_auc_score(y, p_cal))
    ap_cal  = float(average_precision_score(y, p_cal))
else:
    auc_raw = ap_raw = auc_cal = ap_cal = float("nan")

print("\n[OOF CLS] raw : AUC=%.5f AP=%.5f" % (auc_raw, ap_raw))
print("[OOF CLS] cal(%s): AUC=%.5f AP=%.5f" % (calib_kind, auc_cal, ap_cal))

# save calibrator
joblib.dump({"kind": calib_kind, "calibrator": calibrator}, OUT_DIR / "calibrator.joblib")

# ----------------------------
# Threshold tuning (vectorized, FAST)
# - tune: thr_forged, thr_mask, min_pred_patches
# Objective (weighted per-fold):
#   if gt_empty: score=1 - pf*(pred_nonempty)
#   if gt_nonempty: score=pf * dice
# where pf = (p_cal >= thr_forged)
# dice computed on grid with threshold thr_mask
# pred_nonempty uses min_pred_patches
# ----------------------------
gt = (oof_gt > 0).astype(np.uint8)                 # [N,GH,GW]
gt_sum = gt.reshape(n_all, -1).sum(axis=1).astype(np.int32)
gt_empty = (gt_sum == 0)

# balanced weights for objective
w_pos2 = (y.sum() + (y==0).sum()) / (2.0 * max(y.sum(), 1))
w_neg2 = (y.sum() + (y==0).sum()) / (2.0 * max((y==0).sum(), 1))
w_obj = np.where(y == 1, w_pos2, w_neg2).astype(np.float32)

thr_p_list = np.linspace(0.05, 0.95, 37).astype(np.float32)
thr_m_list = np.linspace(0.20, 0.80, 31).astype(np.float32)
min_patches_list = np.array([0, 1, 2, 4, 8, 16], dtype=np.int32)

# precompute per-fold masks
fold_ids = sorted(np.unique(oof_fold).tolist())
fold_masks = [(f, (oof_fold == f)) for f in fold_ids]

# precompute dice and pred_sum for each thr_mask
seg = oof_seg.astype(np.float32)  # [N,GH,GW] (float16 -> float32)

dice_by_tm = np.zeros((len(thr_m_list), n_all), dtype=np.float32)
psum_by_tm = np.zeros((len(thr_m_list), n_all), dtype=np.int32)

print("\n[THR] Precomputing dice / pred_sum across thr_mask ...")
t0 = time.time()
for i_tm, tm in enumerate(thr_m_list):
    pred = (seg >= float(tm)).astype(np.uint8)
    ps = pred.reshape(n_all, -1).sum(axis=1).astype(np.int32)
    inter = (pred & gt).reshape(n_all, -1).sum(axis=1).astype(np.int32)
    gs = gt_sum.astype(np.int32)

    dice = np.zeros(n_all, dtype=np.float32)
    both0 = (ps == 0) & (gs == 0)
    dice[both0] = 1.0
    m = ~both0
    dice[m] = (2.0 * inter[m].astype(np.float32)) / (ps[m].astype(np.float32) + gs[m].astype(np.float32) + 1e-6)

    dice_by_tm[i_tm] = dice
    psum_by_tm[i_tm] = ps

dt = time.time() - t0
print(f"[THR] Precompute done in {dt:.1f}s")

best = {
    "score": -1e9,
    "thr_forged": 0.5,
    "thr_mask": 0.5,
    "min_pred_patches": 0,
}

print("\n[THR] Grid search thr_forged x thr_mask x min_pred_patches (weighted fold-mean) ...")
t0 = time.time()

for tp in thr_p_list:
    pf = (p_cal >= float(tp))  # [N] bool
    for i_tm, tm in enumerate(thr_m_list):
        dice = dice_by_tm[i_tm]   # [N]
        ps   = psum_by_tm[i_tm]   # [N] int

        for mp in min_patches_list:
            pred_nonempty = (ps >= int(mp))  # [N] bool

            # score per sample:
            # gt_empty: 1 - pf*pred_nonempty
            # gt_nonempty: pf*dice
            s = np.zeros(n_all, dtype=np.float32)
            s[gt_empty] = 1.0 - (pf[gt_empty] & pred_nonempty[gt_empty]).astype(np.float32)
            ne = ~gt_empty
            s[ne] = (pf[ne].astype(np.float32) * dice[ne].astype(np.float32))

            # fold mean weighted
            fold_scores = []
            for f, m in fold_masks:
                if m.sum() == 0:
                    continue
                ww = w_obj[m]
                fold_scores.append(float(np.sum(s[m] * ww) / (np.sum(ww) + 1e-12)))
            mean_score = float(np.mean(fold_scores)) if fold_scores else -1e9

            if mean_score > best["score"]:
                best = {
                    "score": mean_score,
                    "thr_forged": float(tp),
                    "thr_mask": float(tm),
                    "min_pred_patches": int(mp),
                }

print(f"[THR] Search done in {time.time()-t0:.1f}s")
print("Best thresholds:")
print(json.dumps(best, indent=2))

# ----------------------------
# Save artifacts
# ----------------------------
thresholds = {
    "thr_forged": best["thr_forged"],
    "thr_p": best["thr_forged"],          # alias for compatibility
    "thr_mask": best["thr_mask"],
    "min_pred_patches": best["min_pred_patches"],
    "grid_hw": [int(GH), int(GW)],
    "img_size": int(IMG_SIZE),
    "patch_size": int(PATCH),
    "calibration": calib_kind,
}
with open(OUT_DIR / "thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)

model_config = {
    "dino_dir": str(CFG["dino_dir"]),
    "img_size": int(IMG_SIZE),
    "patch_size": int(PATCH),
    "grid_hw": [int(GH), int(GW)],
    "unfreeze_last_n_blocks": int(CFG["unfreeze_last_n_blocks"]),
    "note": "Checkpoints store trainable_state only (heads + unfrozen blocks). Load base DINO from dino_dir, then apply trainable_state.",
    "fold_checkpoints": best_fold_paths,
}
with open(OUT_DIR / "model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)

# OOF audit file
df_oof = pd.DataFrame({
    "sample_id": oof_sid,
    "fold": oof_fold,
    "y_forged": oof_y,
    "p_raw": p_raw.astype(np.float32),
    "p_cal": p_cal.astype(np.float32),
    "gt_sum": gt_sum.astype(np.int32),
})
df_oof.to_csv(OUT_DIR / "oof_predictions.csv", index=False)

# final report
report = {
    "cfg": CFG,
    "out_dir": str(OUT_DIR),
    "device": str(device),
    "n_train": int(n_all),
    "forged_rate": float(oof_y.mean()),
    "oof_auc_raw": auc_raw if np.isfinite(auc_raw) else None,
    "oof_ap_raw": ap_raw if np.isfinite(ap_raw) else None,
    "oof_auc_cal": auc_cal if np.isfinite(auc_cal) else None,
    "oof_ap_cal": ap_cal if np.isfinite(ap_cal) else None,
    "thresholds": thresholds,
    "fold_reports": fold_reports,
}
with open(OUT_DIR / "report.json", "w") as f:
    json.dump(report, f, indent=2)

# print checkpoint sizes
sizes = {}
for k, p in best_fold_paths.items():
    pp = Path(p)
    if pp.exists():
        sizes[str(k)] = float(pp.stat().st_size / (1024**2))
print("\nCheckpoint sizes (MB) per fold:", sizes)

print("\nSAVED ->", OUT_DIR)
print("Files:", sorted([p.name for p in OUT_DIR.iterdir()]))

DINO_MT_MODEL_DIR = str(OUT_DIR)
print("\nDONE. Exported: DINO_MT_MODEL_DIR =", DINO_MT_MODEL_DIR)


# Inference Strategy: Two-Pass + Smart Ensemble + Export RLE (Strict Guard)

In [ ]:
# ============================================================
# STAGE 6 — DINOv2 Multi-Task Inference: Two-Pass + Smart Ensemble + Export RLE (Strict Guard)
# ONE CELL, REVISI FULL v3 (FAST + ROBUST + STRICT ORDER)
#
# Requirements:
# - Output STAGE 5 (DINO Multi-task) exists:
#   * DINO_MT_MODEL_DIR (or auto-find /kaggle/working/recodai_luc/models/dinov2_mt_v5_*)
#   * model_config.json, thresholds.json, calibrator.joblib, checkpoints/fold_*.pt
#
# Key upgrades:
# - Mean-weights ensemble across folds (FAST). Optional exact fold ensemble if needed.
# - PASS-1 small (448) + PASS-2 large (700) only for borderline.
# - Strict guard uses: thr_forged + thr_mask + min_pred_patches + anti-huge/fragment rules.
# - Cache PASS-1/PASS-2 masks as NPZ (mask_pack) => rerun is fast.
# - Export RLE strictly in sample_submission order.
# ============================================================

import os, gc, json, time, math, re
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
import joblib

# ----------------------------
# Optional SciPy for morphology / components
# ----------------------------
try:
    import scipy.ndimage as ndi
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ----------------------------
# PATHS (fixed)
# ----------------------------
DATA_ROOT = Path("/kaggle/input/recodai-luc-scientific-image-forgery-detection")
TEST_IMAGES_DIR = DATA_ROOT / "test_images"
SAMPLE_SUB_PATH = DATA_ROOT / "sample_submission.csv"

# ----------------------------
# CACHE DIRS
# ----------------------------
P1_DIR = Path("/kaggle/working/recodai_luc/cache/dino_mt_p1")
P2_DIR = Path("/kaggle/working/recodai_luc/cache/dino_mt_p2")
P1_DIR.mkdir(parents=True, exist_ok=True)
P2_DIR.mkdir(parents=True, exist_ok=True)

OUT_SUB_PATH  = Path("/kaggle/working/submission.csv")
OUT_COPY_PATH = Path("/kaggle/working/recodai_luc/outputs/submission.csv")
OUT_COPY_PATH.parent.mkdir(parents=True, exist_ok=True)

# ----------------------------
# USER TUNABLE (safe defaults)
# ----------------------------
PASS1_SIZE = 448          # must be multiple of patch_size=14 (OK)
PASS2_SIZE = 700          # must be multiple of 14 (OK)
PASS1_BS   = 8            # auto-reduce if OOM
PASS2_BS   = 4
USE_PASS2  = True

BORDER_MARGIN = 0.08      # borderline if |p-thr| <= margin
MAX_BORDERLINE = None     # set int to cap PASS-2 (e.g., 1500)

# FAST ensemble: average weights across folds (recommended)
EXACT_FOLD_ENSEMBLE = False  # if True: run all folds and average outputs (slower)

# RLE order (if you already set global RLE_ORDER, it will use it)
RLE_ORDER = globals().get("RLE_ORDER", "F")
if RLE_ORDER not in ("F","C"):
    RLE_ORDER = "F"

# ----------------------------
# Require sample submission
# ----------------------------
if not SAMPLE_SUB_PATH.exists():
    raise FileNotFoundError(f"sample_submission.csv not found: {SAMPLE_SUB_PATH}")

df_sample = pd.read_csv(SAMPLE_SUB_PATH)
if not {"case_id","annotation"}.issubset({c.lower() for c in df_sample.columns}):
    raise ValueError(f"sample_submission must contain case_id, annotation. Found: {list(df_sample.columns)}")

col_case = [c for c in df_sample.columns if c.lower()=="case_id"][0]
col_ann  = [c for c in df_sample.columns if c.lower()=="annotation"][0]
df_sample = df_sample.rename(columns={col_case:"case_id", col_ann:"annotation"}).copy()
df_sample["case_id"] = df_sample["case_id"].astype(str)

# ----------------------------
# Resolve test images in STRICT sample order
# ----------------------------
IMG_EXTS = {".png",".jpg",".jpeg",".tif",".tiff",".bmp",".webp"}

def build_caseid_map(folder: Path) -> dict:
    mp = {}
    files = [p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    files.sort()
    for p in files:
        cid = p.stem
        if cid not in mp:
            mp[cid] = p
    return mp

test_img_map = build_caseid_map(TEST_IMAGES_DIR)
df_test = pd.DataFrame({"case_id": df_sample["case_id"].astype(str).tolist()})
df_test["image_path"] = df_test["case_id"].map(lambda x: str(test_img_map.get(str(x), "")))

n_ok = int(df_test["image_path"].map(lambda p: Path(p).exists()).sum())
print(f"Test images resolved: {n_ok:,}/{len(df_test):,}")

# ----------------------------
# RLE encode
# ----------------------------
def rle_encode(mask: np.ndarray, order: str="F") -> str:
    m = (mask > 0).astype(np.uint8)
    if m.sum() == 0:
        return ""
    if order.upper() == "F":
        pixels = m.T.reshape(-1)
    else:
        pixels = m.reshape(-1)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[0::2]
    return " ".join(map(str, runs))

# ----------------------------
# Auto-find DINO_MT_MODEL_DIR
# ----------------------------
def _auto_find_latest_dir(root: Path, pattern: str):
    root = Path(root)
    if not root.exists():
        return None
    cands = [p for p in root.glob(pattern) if p.is_dir()]
    if not cands:
        return None
    cands.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]

if "DINO_MT_MODEL_DIR" in globals():
    DINO_MT_MODEL_DIR = Path(str(globals()["DINO_MT_MODEL_DIR"]))
else:
    DINO_MT_MODEL_DIR = _auto_find_latest_dir(Path("/kaggle/working/recodai_luc/models"), "dinov2_mt_v5_*")

if DINO_MT_MODEL_DIR is None or (not DINO_MT_MODEL_DIR.exists()):
    raise FileNotFoundError("DINO_MT_MODEL_DIR not found. Jalankan STAGE 5 DINO Multi-task dulu.")

cfg_path = DINO_MT_MODEL_DIR / "model_config.json"
thr_path = DINO_MT_MODEL_DIR / "thresholds.json"
cal_path = DINO_MT_MODEL_DIR / "calibrator.joblib"
ckpt_dir = DINO_MT_MODEL_DIR / "checkpoints"

for p in [cfg_path, thr_path, cal_path, ckpt_dir]:
    if not p.exists():
        raise FileNotFoundError(f"Missing artifact: {p}")

model_cfg = json.loads(cfg_path.read_text())
thr_cfg   = json.loads(thr_path.read_text())
cal_pack  = joblib.load(cal_path)

calib_kind = cal_pack.get("kind", "none")
calibrator = cal_pack.get("calibrator", None)

# thresholds from stage 5
thr_forged = float(thr_cfg.get("thr_forged", thr_cfg.get("thr_p", 0.5)))
thr_mask   = float(thr_cfg.get("thr_mask", 0.5))
min_pred_patches = int(thr_cfg.get("min_pred_patches", 0))

patch_size = int(thr_cfg.get("patch_size", model_cfg.get("patch_size", 14)))
assert PASS1_SIZE % patch_size == 0 and PASS2_SIZE % patch_size == 0, "PASS sizes must be multiple of patch_size."

print("\nLoaded DINO multi-task artifacts:")
print(f"  model_dir          : {DINO_MT_MODEL_DIR}")
print(f"  dino_dir           : {model_cfg.get('dino_dir')}")
print(f"  patch_size         : {patch_size}")
print(f"  thr_forged         : {thr_forged}")
print(f"  thr_mask           : {thr_mask}")
print(f"  min_pred_patches   : {min_pred_patches}")
print(f"  calib_kind         : {calib_kind}")
print(f"  EXACT_FOLD_ENSEMBLE: {EXACT_FOLD_ENSEMBLE}")
print(f"  RLE_ORDER          : {RLE_ORDER}")

def apply_calibrator(p):
    p = np.asarray(p, dtype=np.float32)
    if calibrator is None or calib_kind == "none":
        return p
    if calib_kind == "isotonic":
        return calibrator.transform(p).astype(np.float32)
    return calibrator.predict_proba(p.reshape(-1,1))[:,1].astype(np.float32)

# ----------------------------
# Build DINO multitask model (same heads as STAGE 5)
# ----------------------------
IMNET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(3,1,1)
IMNET_STD  = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(3,1,1)

def freeze_all(model):
    for p in model.parameters():
        p.requires_grad = False

class DinoMultiTask(nn.Module):
    def __init__(self, backbone: nn.Module, patch: int):
        super().__init__()
        self.backbone = backbone
        self.patch = int(patch)

        # infer embed dim
        with torch.no_grad():
            dummy = torch.zeros((1,3,PASS1_SIZE,PASS1_SIZE), dtype=torch.float32)
            out = self.backbone(pixel_values=dummy)
            D = int(out.last_hidden_state.shape[-1])
        self.embed_dim = D

        self.cls_head = nn.Sequential(nn.LayerNorm(D), nn.Linear(D, 1))
        self.seg_head = nn.Sequential(nn.LayerNorm(D), nn.Linear(D, 1))

    def forward(self, x):
        out = self.backbone(pixel_values=x)
        h = out.last_hidden_state  # [B,1+N,D]
        cls_tok = h[:, 0, :]
        ptok = h[:, 1:, :]
        cls_logit = self.cls_head(cls_tok).squeeze(-1)      # [B]
        seg_patch = self.seg_head(ptok).squeeze(-1)         # [B,N]

        # infer grid from input size (square)
        B, _, H, W = x.shape
        gh = H // self.patch
        gw = W // self.patch
        Nexp = gh * gw
        if seg_patch.shape[1] < Nexp:
            # safety: pad
            pad = Nexp - seg_patch.shape[1]
            seg_patch = torch.cat([seg_patch, seg_patch.new_zeros((B, pad))], dim=1)
        seg_grid = seg_patch[:, :Nexp].reshape(B, gh, gw)
        return cls_logit, seg_grid

# ----------------------------
# Load fold checkpoints and build ensemble state
# ----------------------------
def list_fold_ckpts(ckpt_dir: Path):
    cands = sorted(ckpt_dir.glob("fold_*.pt"))
    if not cands:
        raise FileNotFoundError(f"No fold checkpoints found in {ckpt_dir}")
    return cands

fold_ckpts = list_fold_ckpts(ckpt_dir)
print(f"\nFound fold checkpoints: {len(fold_ckpts)}")

def load_trainable_state(pt_path: Path):
    ck = torch.load(pt_path, map_location="cpu")
    sd = ck.get("trainable_state", None)
    if sd is None:
        raise RuntimeError(f"Checkpoint missing trainable_state: {pt_path}")
    # ensure float32 for averaging
    out = {}
    for k, v in sd.items():
        if torch.is_tensor(v) and v.is_floating_point():
            out[k] = v.float()
        else:
            out[k] = v
    return out

def average_states(states: list):
    keys = set(states[0].keys())
    for st in states[1:]:
        keys &= set(st.keys())
    keys = sorted(list(keys))
    avg = {}
    for k in keys:
        vs = [st[k] for st in states]
        if torch.is_tensor(vs[0]) and vs[0].is_floating_point():
            s = vs[0].clone()
            for t in vs[1:]:
                s += t
            avg[k] = (s / float(len(vs)))
        else:
            avg[k] = vs[0]
    return avg

def apply_trainable_state(model: nn.Module, trainable_state: dict):
    cur = model.state_dict()
    for k, v in trainable_state.items():
        if k in cur:
            if torch.is_tensor(v) and torch.is_tensor(cur[k]):
                cur[k] = v.to(cur[k].dtype)
            else:
                cur[k] = v
    model.load_state_dict(cur, strict=False)

# ----------------------------
# Image preprocessing (square, consistent with STAGE 5)
# ----------------------------
def load_image_square(path: str, size: int):
    p = Path(path)
    if not p.exists():
        return None, None
    im = Image.open(p).convert("RGB")
    orig_w, orig_h = im.size
    if im.size != (size, size):
        im = im.resize((size, size), resample=Image.BICUBIC)
    arr = (np.asarray(im, dtype=np.float32) / 255.0)  # HWC
    x = torch.from_numpy(arr).permute(2,0,1).contiguous()  # CHW
    x = (x - IMNET_MEAN) / IMNET_STD
    meta = {"orig_h": int(orig_h), "orig_w": int(orig_w), "side": int(size)}
    return x, meta

def grid_to_mask_orig(grid_u8: np.ndarray, meta: dict, patch: int):
    # grid_u8: [gh,gw] 0/1 on square side meta["side"]
    side = int(meta["side"])
    orig_h = int(meta["orig_h"])
    orig_w = int(meta["orig_w"])
    mask_sq = np.kron(grid_u8.astype(np.uint8), np.ones((patch, patch), dtype=np.uint8))
    mask_sq = mask_sq[:side, :side]
    # resize square->orig
    if (orig_w, orig_h) != (side, side):
        im = Image.fromarray((mask_sq * 255).astype(np.uint8))
        im = im.resize((orig_w, orig_h), resample=Image.NEAREST)
        mask = (np.array(im) > 0).astype(np.uint8)
    else:
        mask = mask_sq.astype(np.uint8)
    return mask

def pack_mask(mask_u8: np.ndarray):
    mh, mw = mask_u8.shape
    pack = np.packbits((mask_u8 > 0).astype(np.uint8), axis=None).astype(np.uint8)
    return pack, mh, mw

def unpack_mask(pack: np.ndarray, h: int, w: int):
    if h <= 0 or w <= 0 or pack is None or pack.size == 0:
        return np.zeros((max(h,1), max(w,1)), dtype=np.uint8)
    bits = np.unpackbits(pack.astype(np.uint8), axis=None)[: h*w]
    return bits.reshape(h, w).astype(np.uint8)

# ----------------------------
# Post-filter mask (anti-fragment, anti-noise)
# ----------------------------
def filter_mask(mask_u8: np.ndarray, min_area_frac=0.0, keep_topk=10, close_ks=3, open_ks=0):
    if mask_u8.sum() == 0:
        return mask_u8, {"n_comp": 0, "largest": 0}

    H, W = mask_u8.shape
    area = int(mask_u8.sum())
    denom = float(H*W) + 1e-9
    if float(min_area_frac) > 0 and (area/denom) < float(min_area_frac):
        return np.zeros_like(mask_u8, dtype=np.uint8), {"n_comp": 0, "largest": 0}

    m = mask_u8.astype(bool)
    if _HAS_SCIPY:
        if int(close_ks) and int(close_ks) > 1:
            st = np.ones((int(close_ks), int(close_ks)), dtype=bool)
            m = ndi.binary_closing(m, structure=st)
        if int(open_ks) and int(open_ks) > 1:
            st = np.ones((int(open_ks), int(open_ks)), dtype=bool)
            m = ndi.binary_opening(m, structure=st)

        lab, n = ndi.label(m)
        if n <= 0:
            return np.zeros_like(mask_u8, dtype=np.uint8), {"n_comp": 0, "largest": 0}

        areas = np.bincount(lab.ravel())
        areas[0] = 0
        comps = np.where(areas > 0)[0]
        comps = comps[np.argsort(areas[comps])[::-1]]
        if keep_topk and int(keep_topk) > 0:
            comps = comps[:int(keep_topk)]
        out = np.isin(lab, comps).astype(np.uint8)
        largest = int(areas[comps[0]]) if comps.size else 0
        return out, {"n_comp": int(comps.size), "largest": largest}

    # no scipy: minimal
    return mask_u8.astype(np.uint8), {"n_comp": 1, "largest": int(mask_u8.sum())}

# ----------------------------
# Strict guard + quality score (DINO multi-task)
# ----------------------------
def strict_guard(p_cal: float, pred_patches: int, area_frac: float):
    if float(p_cal) < thr_forged:
        return False
    if int(pred_patches) < int(min_pred_patches):
        return False
    # anti-huge mask unless very confident
    if float(area_frac) > 0.65 and float(p_cal) < (thr_forged + 0.12):
        return False
    # tiny mask with low confidence -> reject
    if float(area_frac) < 0.00015 and float(p_cal) < (thr_forged + 0.10):
        return False
    return True

def quality_score(p_cal: float, area_frac: float, n_comp: float, mean_conf_in_mask: float):
    # higher is better
    frag_pen = min(float(n_comp) / 35.0, 1.0) * 0.35
    huge_pen = 0.0
    if float(area_frac) > 0.35:
        huge_pen = min((float(area_frac) - 0.35) / 0.35, 1.0) * 0.45
    q = float(p_cal) * (0.55 + 0.45*float(mean_conf_in_mask)) * (1.0 - frag_pen) * (1.0 - huge_pen)
    return float(q)

def mean_conf_inside(seg_prob_grid: np.ndarray, grid_bin: np.ndarray):
    # seg_prob_grid float32 [gh,gw], grid_bin uint8
    if grid_bin.sum() == 0:
        return 0.0
    v = seg_prob_grid[grid_bin > 0]
    return float(v.mean()) if v.size else 0.0

# ----------------------------
# Cache IO (store final thresholded mask + stats)
# ----------------------------
def cache_path(cache_dir: Path, case_id: str, tag: str):
    return cache_dir / f"{case_id}_{tag}.npz"

def load_cached(cache_dir: Path, case_id: str, tag: str):
    p = cache_path(cache_dir, case_id, tag)
    if not p.exists():
        return None
    z = np.load(p, allow_pickle=False)
    pack = z["mask_pack"].astype(np.uint8).reshape(-1)
    mh = int(z["mask_h"]); mw = int(z["mask_w"])
    info = {k: float(z[k]) for k in z.files if k not in ("mask_pack","mask_h","mask_w")}
    return pack, mh, mw, info

def save_cached(cache_dir: Path, case_id: str, tag: str, mask_u8: np.ndarray, info: dict):
    pack, mh, mw = pack_mask(mask_u8)
    payload = {
        "mask_pack": pack,
        "mask_h": np.int32(mh),
        "mask_w": np.int32(mw),
        "p_raw": np.float32(float(info.get("p_raw", 0.0))),
        "p_cal": np.float32(float(info.get("p_cal", 0.0))),
        "pred_patches": np.float32(float(info.get("pred_patches", 0.0))),
        "area_frac": np.float32(float(info.get("area_frac", 0.0))),
        "n_comp": np.float32(float(info.get("n_comp", 0.0))),
        "mean_conf": np.float32(float(info.get("mean_conf", 0.0))),
        "side": np.float32(float(info.get("side", 0.0))),
    }
    np.savez_compressed(cache_path(cache_dir, case_id, tag), **payload)

# ----------------------------
# Inference core (supports mean-state or exact fold ensemble)
# ----------------------------
def build_model_on_device(dino_dir: str, device: torch.device):
    backbone = AutoModel.from_pretrained(str(dino_dir), local_files_only=True).eval()
    freeze_all(backbone)
    model = DinoMultiTask(backbone, patch=patch_size).eval().to(device)
    return model

def infer_batch(model, x_batch, use_amp=True):
    device = next(model.parameters()).device
    x_batch = x_batch.to(device, non_blocking=True)
    with torch.inference_mode():
        if use_amp and device.type == "cuda":
            with torch.cuda.amp.autocast(True):
                cls_logit, seg_logit = model(x_batch)
        else:
            cls_logit, seg_logit = model(x_batch)
    p_raw = torch.sigmoid(cls_logit).detach().float().cpu().numpy().astype(np.float32)
    seg_prob = torch.sigmoid(seg_logit).detach().float().cpu().numpy().astype(np.float32)  # [B,gh,gw]
    return p_raw, seg_prob

def infer_with_mean_state(image_paths, metas, side: int, batch_size: int, cache_dir: Path, tag: str):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model_on_device(model_cfg["dino_dir"], device)

    # build mean state once
    states = [load_trainable_state(p) for p in fold_ckpts]
    mean_state = average_states(states)
    apply_trainable_state(model, mean_state)

    # optional: half on cuda
    if device.type == "cuda":
        model = model.half()

    n = len(image_paths)
    p_raw_all = np.zeros(n, dtype=np.float32)
    seg_prob_all = None  # store per item while saving cache

    # process in batches
    i = 0
    while i < n:
        j = min(i + batch_size, n)
        xb = []
        mb = metas[i:j]
        for k in range(i, j):
            x, _ = load_image_square(image_paths[k], size=side)
            if x is None:
                x = torch.zeros((3, side, side), dtype=torch.float32)
            xb.append(x)
        xb = torch.stack(xb, dim=0)
        if device.type == "cuda":
            xb = xb.half()
        p_raw, seg_prob = infer_batch(model, xb, use_amp=True)
        p_raw_all[i:j] = p_raw

        # per item: threshold seg -> mask -> post-filter -> cache
        p_cal = apply_calibrator(p_raw)

        gh = side // patch_size
        gw = side // patch_size
        for t in range(j - i):
            meta = mb[t]
            cid = str(meta["case_id"])
            orig_h = int(meta["orig_h"]); orig_w = int(meta["orig_w"])
            seg = seg_prob[t]  # [gh,gw]
            grid = (seg >= thr_mask).astype(np.uint8)
            pred_patches = int(grid.sum())

            mask = grid_to_mask_orig(grid, meta={"orig_h": orig_h, "orig_w": orig_w, "side": side}, patch=patch_size)

            # post-filter: keep stable; set minimal tiny filter
            mask2, comp = filter_mask(mask, min_area_frac=0.0, keep_topk=10, close_ks=3, open_ks=0)

            area = float(mask2.sum())
            area_frac = float(area / (float(orig_h*orig_w) + 1e-9))
            mean_conf = mean_conf_inside(seg, grid)

            info = {
                "p_raw": float(p_raw[t]),
                "p_cal": float(p_cal[t]),
                "pred_patches": float(pred_patches),
                "area_frac": float(area_frac),
                "n_comp": float(comp.get("n_comp", 0)),
                "mean_conf": float(mean_conf),
                "side": float(side),
            }
            save_cached(cache_dir, cid, tag, mask2.astype(np.uint8), info)

        i = j
        if (i % max(100, batch_size*25)) == 0:
            print(f"[{tag}] infer+cache {i:,}/{n:,}")

    # cleanup
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def infer_with_exact_fold_ensemble(image_paths, metas, side: int, batch_size: int, cache_dir: Path, tag: str):
    # exact: average outputs across folds (slower)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model_on_device(model_cfg["dino_dir"], device)
    if device.type == "cuda":
        model = model.half()

    n = len(image_paths)
    # accumulator
    p_raw_sum = np.zeros(n, dtype=np.float32)
    seg_sum = np.zeros((n, side//patch_size, side//patch_size), dtype=np.float32)

    for fi, ckpt_path in enumerate(fold_ckpts, 1):
        st = load_trainable_state(ckpt_path)
        apply_trainable_state(model, st)

        i = 0
        while i < n:
            j = min(i + batch_size, n)
            xb = []
            mb = metas[i:j]
            for k in range(i, j):
                x, _ = load_image_square(image_paths[k], size=side)
                if x is None:
                    x = torch.zeros((3, side, side), dtype=torch.float32)
                xb.append(x)
            xb = torch.stack(xb, dim=0)
            if device.type == "cuda":
                xb = xb.half()
            p_raw, seg_prob = infer_batch(model, xb, use_amp=True)
            p_raw_sum[i:j] += p_raw
            seg_sum[i:j] += seg_prob
            i = j

        print(f"[{tag}] fold {fi}/{len(fold_ckpts)} done")

    p_raw_avg = p_raw_sum / float(len(fold_ckpts))
    seg_avg = seg_sum / float(len(fold_ckpts))
    p_cal = apply_calibrator(p_raw_avg)

    # write cache
    gh = side // patch_size
    gw = side // patch_size
    for idx in range(n):
        meta = metas[idx]
        cid = str(meta["case_id"])
        orig_h = int(meta["orig_h"]); orig_w = int(meta["orig_w"])
        seg = seg_avg[idx]
        grid = (seg >= thr_mask).astype(np.uint8)
        pred_patches = int(grid.sum())

        mask = grid_to_mask_orig(grid, meta={"orig_h": orig_h, "orig_w": orig_w, "side": side}, patch=patch_size)
        mask2, comp = filter_mask(mask, min_area_frac=0.0, keep_topk=10, close_ks=3, open_ks=0)

        area = float(mask2.sum())
        area_frac = float(area / (float(orig_h*orig_w) + 1e-9))
        mean_conf = mean_conf_inside(seg, grid)

        info = {
            "p_raw": float(p_raw_avg[idx]),
            "p_cal": float(p_cal[idx]),
            "pred_patches": float(pred_patches),
            "area_frac": float(area_frac),
            "n_comp": float(comp.get("n_comp", 0)),
            "mean_conf": float(mean_conf),
            "side": float(side),
        }
        save_cached(cache_dir, cid, tag, mask2.astype(np.uint8), info)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ----------------------------
# Build meta list + load from cache (PASS-1)
# ----------------------------
t0 = time.time()

metas = []
need_p1 = []
for i, row in df_test.iterrows():
    cid = str(row["case_id"])
    ip = str(row["image_path"])
    if not Path(ip).exists():
        metas.append({"case_id": cid, "image_path": ip, "orig_h": 1, "orig_w": 1})
        continue
    im = Image.open(ip)
    ow, oh = im.size
    im.close()
    metas.append({"case_id": cid, "image_path": ip, "orig_h": int(oh), "orig_w": int(ow)})

    c = load_cached(P1_DIR, cid, tag=f"s{PASS1_SIZE}")
    if c is None:
        need_p1.append(i)

print(f"\nPASS-1 cache check: need_compute={len(need_p1):,}/{len(df_test):,}")

# Compute missing PASS-1 caches
if len(need_p1) > 0:
    idxs = need_p1
    img_paths = [metas[i]["image_path"] for i in idxs]
    meta_sub  = [dict(metas[i], case_id=str(metas[i]["case_id"])) for i in idxs]
    # batch size auto reduce on OOM
    bs = int(PASS1_BS)
    while True:
        try:
            if EXACT_FOLD_ENSEMBLE:
                infer_with_exact_fold_ensemble(img_paths, meta_sub, side=PASS1_SIZE, batch_size=bs, cache_dir=P1_DIR, tag=f"s{PASS1_SIZE}")
            else:
                infer_with_mean_state(img_paths, meta_sub, side=PASS1_SIZE, batch_size=bs, cache_dir=P1_DIR, tag=f"s{PASS1_SIZE}")
            break
        except RuntimeError as e:
            if "out of memory" in str(e).lower() and bs > 1:
                bs = max(1, bs // 2)
                print(f"[OOM] Reduce PASS1_BS -> {bs}")
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue
            raise

# Load PASS-1 info for all
p1_pack = [None]*len(df_test)
p1_hw   = [None]*len(df_test)
p1_info = [None]*len(df_test)

miss_p1 = 0
for i, row in df_test.iterrows():
    cid = str(row["case_id"])
    c = load_cached(P1_DIR, cid, tag=f"s{PASS1_SIZE}")
    if c is None:
        miss_p1 += 1
        p1_pack[i] = np.zeros((0,), dtype=np.uint8)
        p1_hw[i]   = (1,1)
        p1_info[i] = {"p_raw":0.0,"p_cal":0.0,"pred_patches":0.0,"area_frac":0.0,"n_comp":0.0,"mean_conf":0.0,"side":float(PASS1_SIZE)}
    else:
        pack, mh, mw, info = c
        p1_pack[i] = pack
        p1_hw[i]   = (mh, mw)
        p1_info[i] = info

print(f"PASS-1 loaded. missing_after_compute={miss_p1:,}")

# ----------------------------
# Borderline selection for PASS-2
# ----------------------------
borderline = []
if USE_PASS2:
    for i, row in df_test.iterrows():
        cid = str(row["case_id"])
        ip  = str(row["image_path"])
        if not Path(ip).exists():
            continue

        p = float(p1_info[i].get("p_cal", 0.0))
        area = float(p1_info[i].get("area_frac", 0.0))
        pp = int(round(float(p1_info[i].get("pred_patches", 0.0))))
        # borderline conditions (smart)
        near_thr = (abs(p - thr_forged) <= float(BORDER_MARGIN))
        repair_small = (p >= (thr_forged - 0.06)) and (pp < max(min_pred_patches, 2)) and (area < 0.0012)
        recover_fn = (p >= (thr_forged - 0.10)) and (pp >= max(min_pred_patches-1, 1)) and (area >= 0.00025)
        if (near_thr or repair_small or recover_fn):
            borderline.append((cid, abs(p - thr_forged)))

    borderline.sort(key=lambda x: x[1])
    if MAX_BORDERLINE is not None and len(borderline) > int(MAX_BORDERLINE):
        borderline = borderline[:int(MAX_BORDERLINE)]
    borderline_ids = [x[0] for x in borderline]
else:
    borderline_ids = []

print(f"Borderline for PASS-2: {len(borderline_ids):,}/{len(df_test):,}")

# Compute missing PASS-2 caches for borderline only
if USE_PASS2 and len(borderline_ids) > 0:
    need_p2 = []
    id_to_index = {str(df_test.iloc[i]["case_id"]): i for i in range(len(df_test))}
    for cid in borderline_ids:
        if load_cached(P2_DIR, cid, tag=f"s{PASS2_SIZE}") is None:
            need_p2.append(cid)

    print(f"PASS-2 cache check: need_compute={len(need_p2):,}/{len(borderline_ids):,}")

    if len(need_p2) > 0:
        idxs = [id_to_index[cid] for cid in need_p2]
        img_paths = [metas[i]["image_path"] for i in idxs]
        meta_sub  = [dict(metas[i], case_id=str(metas[i]["case_id"])) for i in idxs]
        bs = int(PASS2_BS)
        while True:
            try:
                if EXACT_FOLD_ENSEMBLE:
                    infer_with_exact_fold_ensemble(img_paths, meta_sub, side=PASS2_SIZE, batch_size=bs, cache_dir=P2_DIR, tag=f"s{PASS2_SIZE}")
                else:
                    infer_with_mean_state(img_paths, meta_sub, side=PASS2_SIZE, batch_size=bs, cache_dir=P2_DIR, tag=f"s{PASS2_SIZE}")
                break
            except RuntimeError as e:
                if "out of memory" in str(e).lower() and bs > 1:
                    bs = max(1, bs // 2)
                    print(f"[OOM] Reduce PASS2_BS -> {bs}")
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    continue
                raise

# ----------------------------
# Smart ensemble P1 vs P2 + Strict guard + Export
# ----------------------------
def iou_masks(m1: np.ndarray, m2: np.ndarray):
    inter = float(np.logical_and(m1 > 0, m2 > 0).sum())
    uni = float(np.logical_or(m1 > 0, m2 > 0).sum()) + 1e-9
    return inter/uni

results = []
n_pred_forg = 0
n_used_p2 = 0
n_union = 0
n_inter = 0

for i, row in df_test.iterrows():
    cid = str(row["case_id"])
    ip  = str(row["image_path"])

    # PASS-1 info
    info1 = p1_info[i]
    mh1, mw1 = p1_hw[i]
    pcal1 = float(info1.get("p_cal", 0.0))
    area1 = float(info1.get("area_frac", 0.0))
    pp1   = int(round(float(info1.get("pred_patches", 0.0))))
    nc1   = float(info1.get("n_comp", 0.0))
    mc1   = float(info1.get("mean_conf", 0.0))

    g1 = strict_guard(pcal1, pp1, area1)
    q1 = quality_score(pcal1, area1, nc1, mc1)

    final_source = "p1"
    final_guard = g1
    final_mask = None
    final_pcal = pcal1
    final_q = q1

    # PASS-2 available?
    use_p2 = False
    if USE_PASS2 and (cid in set(borderline_ids)) and Path(ip).exists():
        c2 = load_cached(P2_DIR, cid, tag=f"s{PASS2_SIZE}")
        if c2 is not None:
            use_p2 = True
            pack2, mh2, mw2, info2 = c2
            pcal2 = float(info2.get("p_cal", 0.0))
            area2 = float(info2.get("area_frac", 0.0))
            pp2   = int(round(float(info2.get("pred_patches", 0.0))))
            nc2   = float(info2.get("n_comp", 0.0))
            mc2   = float(info2.get("mean_conf", 0.0))

            g2 = strict_guard(pcal2, pp2, area2)
            q2 = quality_score(pcal2, area2, nc2, mc2)

            # decision rules (stronger + stable)
            if g2 and (not g1):
                final_source = "p2"; final_guard = True; final_pcal = pcal2; final_q = q2
                final_mask = unpack_mask(pack2, mh2, mw2)
                n_used_p2 += 1
            elif g1 and (not g2):
                final_source = "p1"
            else:
                if g1 and g2:
                    # compare masks
                    m1 = unpack_mask(p1_pack[i], mh1, mw1)
                    m2 = unpack_mask(pack2, mh2, mw2)
                    iou = iou_masks(m1, m2)

                    strong1 = (pcal1 >= thr_forged + 0.12) and (mc1 >= 0.60)
                    strong2 = (pcal2 >= thr_forged + 0.12) and (mc2 >= 0.60)

                    if iou >= 0.18 and strong1 and strong2:
                        mu = np.logical_or(m1 > 0, m2 > 0).astype(np.uint8)
                        final_source = "union"; final_guard = True; final_pcal = max(pcal1, pcal2); final_q = max(q1, q2)
                        final_mask = mu
                        n_union += 1
                        n_used_p2 += 1
                    elif iou <= 0.06 and strong1 and strong2:
                        mi = np.logical_and(m1 > 0, m2 > 0).astype(np.uint8)
                        # re-check guard after intersection
                        area_i = float(mi.sum()) / (float(mi.size) + 1e-9)
                        pp_i = int(round((mi.sum() / (patch_size*patch_size)) / max(1.0, (PASS1_SIZE*PASS1_SIZE)/(patch_size*patch_size))))  # coarse proxy
                        if strict_guard(max(pcal1, pcal2), max(pp1, pp2), area_i):
                            final_source = "inter"; final_guard = True; final_pcal = max(pcal1, pcal2); final_q = max(q1, q2)
                            final_mask = mi
                            n_inter += 1
                            n_used_p2 += 1
                        else:
                            # fallback best quality
                            if q2 > q1 * 1.03:
                                final_source = "p2"; final_guard = g2; final_pcal = pcal2; final_q = q2
                                final_mask = m2
                                n_used_p2 += 1
                            else:
                                final_source = "p1"
                    else:
                        if q2 > q1 * 1.05:
                            final_source = "p2"; final_guard = g2; final_pcal = pcal2; final_q = q2
                            final_mask = m2
                            n_used_p2 += 1
                        else:
                            final_source = "p1"
                else:
                    # both not guarded: pick best quality if clearly better
                    if q2 > q1 * 1.12:
                        final_source = "p2"; final_guard = g2; final_pcal = pcal2; final_q = q2
                        final_mask = unpack_mask(pack2, mh2, mw2)
                        n_used_p2 += 1
                    else:
                        final_source = "p1"

    # final annotation
    if final_guard:
        if final_mask is None:
            final_mask = unpack_mask(p1_pack[i], mh1, mw1)
        rle = rle_encode(final_mask.astype(np.uint8), order=RLE_ORDER)
        ann = rle if rle != "" else "authentic"
        if ann != "authentic":
            n_pred_forg += 1
    else:
        ann = "authentic"

    results.append({"case_id": cid, "annotation": ann})

df_sub = pd.DataFrame(results)
df_sub = df_sample[["case_id"]].merge(df_sub, on="case_id", how="left")
df_sub["annotation"] = df_sub["annotation"].fillna("authentic").astype(str)

df_sub.to_csv(OUT_SUB_PATH, index=False)
df_sub.to_csv(OUT_COPY_PATH, index=False)

dt = time.time() - t0
print("\nDONE.")
print(f"submission.csv -> {OUT_SUB_PATH}")
print(f"copy          -> {OUT_COPY_PATH}")
print(f"elapsed_s     -> {dt:.1f}")
print(f"pred_forged   -> {n_pred_forg:,}/{len(df_sub):,}")
print(f"used_pass2    -> {n_used_p2:,} | union={n_union:,} | inter={n_inter:,}")
print(df_sub.head())
